# 07 · SEI 상(相) 매핑 — halo·ring·spot 픽셀 분류 → phase map + 두께

흐름: **입력 → 전처리 → 개요(median/MAX/center) → radial(예상 peak) → Halo → Ring → Spot → Phase map(확정/예상/약함) → 두께**.
각 픽셀에서 radial integration을 해 halo(비정질)·ring(다결정)·spot(단결정)을 보고, 물질과 상을 **확정/예상/약함**으로 판정한다.
로직은 패키지(`fds.classify_pixels` 등), 이 노트북은 **각 셀 파라미터를 노출**해 데이터마다 조절한다.
플롯 텍스트는 ASCII(한글은 마크다운/print만). 판정: **확정**=스팟 인덱싱(격자 자기일관) / **예상**=링 지문 일치 / **약함**=물질이나 상 불명.

## 1) 입력 — 로드 (경로만 바꾸면 어느 데이터든)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"   # ★데이터셋
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)
DET_BIN    = 1                       # 검출기 비닝(메모리). q_max는 안 변함
Q_UNIT_HINT= "1/nm"                 # dm 단위(0.043888 1/nm)
CANDIDATES = ["LiF","Li2O","Li3N","Li2CO3","Li2S"]
N_JOBS     = -1                      # 병렬 코어(-1=전부; 32코어면 32)
PHASE_COL  = {"LiF":"#2ca02c","Li2O":"#1f77b4","Li3N":"#9467bd","Li2CO3":"#ff7f0e","Li2S":"#8c564b"}
# ★후보 결정 링 = '전자 구조인자'로 계산한 실제 세기(Kirkland f_e, |F_hkl|²). 원자번호 효과(Li 약산란)·소멸 반영.
#   (rough 공칭표 COMPOUND_RINGS 대신 이걸 매칭/그림에 사용. d-spacing은 같고 세기가 물리적으로 바뀜.)
ERINGS     = fds.electron_ring_table(CANDIDATES, d_min=1.0, d_max=5.0, min_w=0.05)

def _synth(Sy=22,Sx=30,H=88,W=88,seed=0):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/8); halo=lambda r0,s=4:np.exp(-(rr-r0)**2/(2*s**2))
    def sp(r0,n=6,a=4):
        im=np.zeros((H,W))
        for k in range(n):
            t=2*np.pi*k/n; im+=a*np.exp(-((xx-cx-r0*np.cos(t))**2+(yy-cy-r0*np.sin(t))**2)/(2*1.6**2))
        return im
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            base=0.9*beam if iy>=Sy-3 else (beam+sp(20)+0.5*halo(20) if ix<Sx//2 else beam+1.2*halo(18))
            cube[iy,ix]=np.clip(base+0.15*rng.standard_normal((H,W)),0,None)
    return cube
if USE_SYNTHETIC: cube=fds.from_array(_synth(),q_per_px=0.02,name="synthetic")
else:
    cube=fds.load(DM4_PATH,Q_UNIT_HINT)
    if DET_BIN>1: cube=fds.bin_cube_detector(cube,DET_BIN)
scan=cube.scan_shape; QPP=cube.calibration.q_per_px
NAME=("synthetic" if USE_SYNTHETIC else os.path.splitext(os.path.basename(DM4_PATH))[0])
SAVE_DIR=("nb7_outputs" if USE_SYNTHETIC else os.path.dirname(DM4_PATH)+"/nb7_outputs"); os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,n): p=os.path.join(SAVE_DIR,f"{NAME}_{n}.png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(n,h,rows):
    import csv; p=os.path.join(SAVE_DIR,f"{NAME}_{n}.csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(h); w.writerows(rows)
    print("saved:",p)
print("cube:",cube.shape,"| q_per_px=",QPP,"| cores:",os.cpu_count(),"->",os.path.abspath(SAVE_DIR))

## 2) 전처리 — 진단(측정 먼저, 필요할 때만 교정)

인공물(중심/wander/타원/defect)은 교정, 신호 평활(블러)은 금지. 여기선 중심과 상태를 진단한다.

In [ ]:
# --- 이 셀 파라미터 ---
HOT_THRESHOLD  = 8.0    # hot/dead 검출 민감도(작을수록 민감). 진짜 스팟이 지워지면 키우기
WANDER_WARN_PX = 1.0    # 이보다 크면 per-position 정렬 권고
ELLIP_WARN     = 0.02   # 타원율 이보다 크면 타원 보정 권고
diag=fds.diagnose_cube(cube,hot_threshold=HOT_THRESHOLD); center=diag["center"]
print("=== 전처리 진단 ===")
print(f"  center           = ({center[0]:.1f},{center[1]:.1f})  (hot-pixel 제거 후 무게중심)")
print(f"  beam wander      = {diag['wander_px']:.2f} px  (>{WANDER_WARN_PX} 이면 정렬 고려)")
print(f"  detector defects = {100*diag['bad_pixel_frac']:.2f} %")
print(f"  ring ellipticity = {100*diag['ellipticity']:.1f} % @ {diag['ellipse_angle_deg']:.0f}deg  (>{100*ELLIP_WARN:.0f}% 면 보정)")
for n in diag["notes"]: print("  -",n)

## 3) 개요 — median / MAX NBD / center

median=비정질 halo가 잘 보임, MAX=다결정 링·스팟이 모여 보임.

In [ ]:
# --- 이 셀 파라미터 ---
CENTER = None       # None=진단값. 수동이면 (cx,cy)
if CENTER is not None: center=CENTER
med=fds.median_pattern(cube); mx=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
fig,ax=plt.subplots(1,2,figsize=(9,4.4))
ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(*center,"c+",ms=9); ax[0].set_title("median NBD (log) - amorphous halo"); ax[0].axis("off")
ax[1].imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="magma"); ax[1].plot(*center,"c+",ms=9); ax[1].set_title("MAX NBD (gamma) - rings/spots"); ax[1].axis("off")
plt.tight_layout(); save(fig,"03_overview"); plt.show()

## 4) Radial integration — **halo 봉우리 검출·확정**(bandpass + convex + gaussian), q·d

halo 위치를 **2단계 배경 제거 후 검출**해 §5로 넘긴다:
1. **convex 배경 빼기**(거친 배경: 빔꼬리+diffuse) → bump
2. **1D bandpass**(좁은평활 − 넓은평활) → bump에 남은 넓은 FSDP 엔벨로프를 벗겨 **어깨·약한 봉우리를 분리**
3. **bandpass에서 find_peaks → 그 자리에 gaussian fit** → 각 봉우리 **정확한 위치**(keep-all: 봉우리 안 버림). 자동이 놓친 게 있으면 **HALO_QS_MANUAL**에 눈으로 본 q를 넣으면 그 자리에 fit.
4. **폭(FWHM)으로 비정질/결정질 분류 (NBD 기준)** → ★NBD에선 봉우리 폭이 grain(Scherrer/DP)이 아니라 **빔 수렴각=디스크 폭**이 지배. **결정질=probe-limited(빔 디스크 폭≈), 비정질=그보다 훨씬 넓음.** 컷은 **측정한 빔 디스크 폭**의 배수(자동). 넓음→비정질(§5), 좁음→결정질(§4b 매칭·§6). L(상관길이)은 비정질에만 유효. FSDP는 항상 비정질(nn≈1.23/q 표기).
5. **2D bandpass 확인** → 각 halo별 NBD(매끈한 radial 배경 빼 뿌연 링) → "진짜 diffuse 링인가" 눈으로.
비정질 halo(q,d)가 §5 프로브. **결정질 봉우리는 §4b 물질검증·§6로.** (§5는 맵만.)

In [ ]:
# --- 이 셀 파라미터 ---
RING_QBEAM=0.20; RING_QMAX=1.0; RING_NSIG=1.5; RING_TOPN=8   # 링 검출
HALO_QLO=0.20               # halo 검출 하한(1/A) — 빔/빔가장자리 배제(★FSDP가 빔쪽이면 낮추기)
HALO_MAXPK=5                # 검출할 halo 최대 개수
HALO_PROM=0.05              # 검출신호(contrast) 대비 최소 prominence. ★약한 halo 안 잡히면 낮추기(0.03), 가짜 많으면 키우기
HALO_MINWIDTH=0.04          # halo=넓은 봉우리만(반폭 1/A). sharp 결정질 링은 배제(→ §6 ring)
HALO_DEG=2                  # convex 배경 차수(2 권장). None이면 rolling-min(어깨 못 잡음)
HALO_QS_MANUAL=None         # ★bump/gaussian 그림 보고 halo를 직접 지정(예: [0.25,0.45,0.65,0.80]). None=자동
BP_WIDE=None; BP_NARROW=None  # bandpass 창(1/A). None=FSDP 폭에서 자동. 숫자 넣으면 수동 덮어쓰기
HALO_BP_PROM=1.0           # bandpass 봉우리 최소 prominence(노이즈 MAD 배수). ★어깨 안 잡히면 낮추기(0.7)
HALO_SHOULDER=0.4          # ★어깨(shoulder) prominence 문턱(노이즈 MAD 배수). 이 값~HALO_BP_PROM 사이의 약한 bandpass 극대 = '어깨'로 표시(낮은 신뢰). 0=끄기
BP_VIEWDQ=0.05              # 각 halo 2D bandpass 볼 때 밴드 반폭(1/A)
meanpat=fds.to_pattern(cube)                                    # mean(전체) — halo가 median보다 잘 남음
qd,Id=fds.azimuthal_integrate(meanpat,center,q_per_px=QPP)     # mean -> halo
qm,Im=fds.azimuthal_integrate(mx,center,q_per_px=QPP)          # max -> rings
# 1) 대략 봉우리(배경 excise용) → 2) convex 배경 뺀 bump → 3) ★bandpass(DoG)에서 봉우리 검출(nb6-1 방식: 어깨·약한 고q까지) → 4) 가우시안 분해
from scipy.ndimage import uniform_filter1d as _u1d
from scipy.signal import find_peaks as _fpk
_rough=fds.amorphous_halo_peaks(qd,Id,q_lo=HALO_QLO,q_hi=RING_QMAX,smooth=3,prominence_frac=0.03,min_width_q=HALO_MINWIDTH,deg=HALO_DEG,max_peaks=HALO_MAXPK)
HALO_BG_BEAM=0.14; HALO_BG_QMAX=1.05; HALO_BG_DEG=2          # bump용 convex 배경(§5와 동일하게)
bg1d=fds.halo_background_1d(qd,Id,_rough or [0.25],dq=0.06,beam_cut=HALO_BG_BEAM,q_max=HALO_BG_QMAX,deg=HALO_BG_DEG)
vsel=(qd>=HALO_BG_BEAM)&(qd<=HALO_BG_QMAX); bump=np.zeros_like(qd); bump[vsel]=np.clip(Id-bg1d,0,None)[vsel]
# FSDP 폭을 데이터에서 자동 측정 → bandpass 창·shoulder 제외를 여기서 유도(고정 0.02/0.14 안 씀)
_dqb=float(np.median(np.diff(qd))); _ix=np.arange(len(qd))
_fw=(qd>HALO_QLO)&(qd<HALO_BG_QMAX)
_fsdp_i=int(_ix[_fw][np.argmax(bump[_fw])]) if _fw.any() else int(np.argmax(bump))      # FSDP = 최강 봉우리
_hm=bump[_fsdp_i]*0.5; _l=_fsdp_i
while _l>0 and bump[_l]>_hm: _l-=1
_r=_fsdp_i
while _r<len(qd)-1 and bump[_r]>_hm: _r+=1
_fwhm=max(float(qd[_r]-qd[_l]),4*_dqb)                                                  # FSDP 반치폭(자동)
_ww=max(3,int((BP_WIDE if BP_WIDE else 1.3*_fwhm)/_dqb))                                # 넓은평활 ≈ 1.3×FSDP폭(자동; 넓으면 고q halo 병합)
_wn=max(1,int((BP_NARROW if BP_NARROW else max(2*_dqb,0.25*_fwhm))/_dqb))               # 좁은평활 ≈ ¼×FSDP폭
# bandpass = log I(q)의 DoG(좁은 - 넓은). ★log이라 FSDP·고q halo가 비슷한 높이로 나옴(고q도 잘 보임)
_lI=np.log(np.clip(Id,1e-3,None)); bp1d=_u1d(_lI,_wn)-_u1d(_lI,_ww)
_nz=1.4826*np.median(np.abs(bp1d[_fw]-np.median(bp1d[_fw])))+1e-9
# ★봉우리 = 1D bandpass의 극대값 그대로(가우시안 재fit·병합 안 함). 각 극대는 3점 포물선으로 sub-bin 보정만.
_fp,_=_fpk(bp1d,prominence=HALO_BP_PROM*_nz,distance=max(3,_wn))
_fp=[int(i) for i in _fp if HALO_QLO<qd[i]<HALO_BG_QMAX]
def _parab(q,y,i):                                             # 3점 포물선 꼭지점(sub-bin), 병합 없음
    if 0<i<len(y)-1:
        den=(y[i-1]-2*y[i]+y[i+1])
        if den!=0: return float(q[i]+0.5*(y[i-1]-y[i+1])/den*(q[i+1]-q[i]))
    return float(q[i])
_maxq=sorted(_parab(qd,bp1d,i) for i in _fp)                   # 주 봉우리(prominence≥HALO_BP_PROM)
# ★어깨(shoulder): bandpass는 어깨도 '약한 극대'로 바꿈 → 주 문턱 미달(HALO_SHOULDER~HALO_BP_PROM)의 약한 극대를 어깨로(낮은 신뢰).
_shq=[]
if HALO_SHOULDER and HALO_SHOULDER<HALO_BP_PROM:
    _ap,_apd=_fpk(bp1d,prominence=HALO_SHOULDER*_nz,distance=max(3,_wn))
    for i,prom in zip(_ap,_apd['prominences']):
        i=int(i)
        if HALO_QLO<qd[i]<HALO_BG_QMAX and prom<HALO_BP_PROM*_nz and not any(abs(i-j)<=2 for j in _fp):
            _shq.append(_parab(qd,bp1d,i))                     # 주 문턱 미달의 약한 극대 = 어깨
auto_peaks=sorted(_maxq+_shq)
print(f"  [자동] FSDP반치폭={_fwhm:.3f} → bandpass 창 narrow={_wn*_dqb:.3f} broad={_ww*_dqb:.3f} | 극대 {len(_maxq)}개"+(f" + 어깨 {len(_shq)}개 q={[round(x,3) for x in _shq]}" if _shq else " (어깨 없음)"))
# 수동 지정(HALO_QS_MANUAL) 있으면 우선, 없으면 극대+어깨 사용
halo_peaks=sorted([float(x) for x in HALO_QS_MANUAL]) if HALO_QS_MANUAL else (auto_peaks or [0.25])
shoulder_qs=set() if HALO_QS_MANUAL else set(round(x,4) for x in _shq)   # 그림서 어깨 구분표시용
halo_q=halo_peaks[0]                                           # FSDP = 최저 q halo(정의)
comps=[(p,1.0/p if p>0 else np.inf,float(np.clip(bp1d,0,None)[int(np.argmin(np.abs(qd-p)))]),0.02) for p in halo_peaks]  # 그림 마커용(위치=bandpass 극대)
# ★NBD 분류: 봉우리 폭은 grain(Scherrer/DP)이 아니라 '빔 수렴각=디스크 폭'이 지배.
#   결정질=probe-limited(디스크 폭≈), 비정질=그보다 훨씬 넓음. 기준 = 측정한 빔 디스크 폭.
BEAM_DISK_Q=None                                              # ★빔 디스크 반경(q,1/A)=수렴각. 알면 직접 입력(제일 정확). None=자동추정
BEAM_MULT=2.5                                                  # 빔 디스크 폭의 이 배수 초과 = 비정질. ★조정 가능
def _fwhm_on(q,prof,q0):
    i0=int(np.argmin(np.abs(q-q0))); pk=float(prof[i0])
    if pk<=0: return np.inf
    h=pk*0.5; a=i0
    while a>0 and prof[a]>h: a-=1
    b=i0
    while b<len(q)-1 and prof[b]>h: b+=1
    return float(q[b]-q[a])
PROBE_THRESH=0.5                                              # 직접빔 디스크 임계값(최대의 배수). py4DSTEM get_probe_size 방식
if BEAM_DISK_Q:
    beam_wq=float(BEAM_DISK_Q); _rpx=beam_wq/QPP              # 사용자 입력
else:                                                         # 직접빔 디스크 반경: 임계값 위 중심 연결영역 면적→반경
    from scipy.ndimage import label as _lab2
    _pn=np.asarray(meanpat,float); _thr=PROBE_THRESH*float(np.nanmax(_pn))
    _mk=_pn>=_thr; _ll,_=_lab2(_mk)
    _cc=int(_ll[int(round(center[1])),int(round(center[0]))])
    _area=int((_ll==_cc).sum()) if _cc>0 else int(_mk.sum())
    _rpx=float(np.sqrt(_area/np.pi)); beam_wq=max(_rpx*QPP,2*_dqb)   # 디스크 반경(q)
cryst_max=BEAM_MULT*beam_wq                                    # 이 이하 폭 = probe-limited = 결정질
print(f"  [직접빔 probe] 디스크 반경 = {_rpx:.1f}px = {beam_wq:.3f} 1/A (결정질 폭 컷 {cryst_max:.3f})"+(" [수동]" if BEAM_DISK_Q else " [측정]"))
# 확인 그림: 직접빔(log) + 측정된 디스크 가장자리 원
_zt=int(min(meanpat.shape[0]//2,meanpat.shape[1]//2,max(_rpx*3,10)))
figp,axp=plt.subplots(1,1,figsize=(4,4))
axp.imshow(np.log1p(np.asarray(meanpat,float)),cmap="magma"); axp.plot(*center,"c+",ms=8)
axp.add_patch(plt.Circle(center,_rpx,fill=False,ec="cyan",ls="--",lw=1.5))
axp.set_xlim(center[0]-_zt,center[0]+_zt); axp.set_ylim(center[1]+_zt,center[1]-_zt)
axp.set_title(f"direct beam (probe) + measured disk\nr={_rpx:.1f}px={beam_wq:.3f}/A (check this edge!)",fontsize=9); axp.axis("off")
plt.tight_layout(); save(figp,"04_probe_disk"); plt.show()
peak_fwhm={p:_fwhm_on(qd,bump,p) for p in halo_peaks}
# L(상관길이)은 '비정질'(넓은 봉우리)에만 유효 — 결정질은 probe-limited라 grain 크기 산출 불가
peak_L={p:(0.9/peak_fwhm[p] if peak_fwhm[p]>cryst_max else np.nan) for p in halo_peaks}
amorph_peaks=[p for p in halo_peaks if peak_fwhm[p]> cryst_max]   # 빔보다 넓음 = 비정질
cryst_peaks =[p for p in halo_peaks if peak_fwhm[p]<=cryst_max]   # 빔 폭 ≈ = probe-limited 결정질
if halo_q not in amorph_peaks and halo_q in halo_peaks:          # FSDP는 항상 비정질(중거리질서)
    amorph_peaks=sorted(set(amorph_peaks)|{halo_q}); cryst_peaks=[p for p in cryst_peaks if p!=halo_q]
print(f"  [분류] 빔디스크폭={beam_wq:.3f} → 결정질 컷={cryst_max:.3f}(1/A) | 비정질 {[round(p,3) for p in amorph_peaks]} | 결정질 {[round(p,3) for p in cryst_peaks]}")
for p in halo_peaks:
    typ='AMORPH' if p in amorph_peaks else 'CRYST'
    _Lstr=(f"corr.L~{peak_L[p]:.0f}A" if typ=='AMORPH' and np.isfinite(peak_L[p]) else "probe-limited(grain 미측정)")
    print(f"    q={p:.3f} d={1/p:.2f}A FWHM={peak_fwhm[p]:.3f} [{typ}] {_Lstr}")
rings_q=fds.detect_rings(mx,center,QPP,q_beam=RING_QBEAM,q_max=RING_QMAX,nsig=RING_NSIG,top_n=RING_TOPN)
print("자동검출(단순 최대) halo: "+(", ".join(f"q{p:.3f}(d{1/p:.2f})" for p in sorted(auto_peaks)) if auto_peaks else "없음"))
print("가우시안 분해 성분: "+(", ".join(f"q{c:.3f}(d{1/c:.2f}A,amp{a:.2f})" for c,d,a,w in comps) if comps else "없음")+(" [seed=수동]" if HALO_QS_MANUAL else " [seed=자동]"))
print(f"→ 사용 halo_peaks = {[round(p,3) for p in halo_peaks]} | FSDP q{halo_q:.3f}(d{1/halo_q:.2f}A) | 링 q="+", ".join(f"{r:.3f}(d{1/r:.2f})" for r in rings_q))
fig,ax=plt.subplots(1,2,figsize=(15,4.3))
# 왼쪽: I(q) + 검출 halo(q,d) + 예상 링
ax[0].semilogy(qm,np.clip(Im,1e-2,None),"k-",lw=0.9,label="MAX I(q)")
ax[0].semilogy(qd,np.clip(Id,1e-2,None),"0.5",lw=0.8,label="mean I(q)")
for cc in CANDIDATES:
    for dd,w in ERINGS[cc]: ax[0].axvline(1/dd,color=PHASE_COL[cc],ls="--",lw=0.4+0.9*w,alpha=0.5)  # 전자세기
for r in rings_q: ax[0].axvline(r,color="k",ls=":",lw=0.7)
for p in halo_peaks:
    _am=p in amorph_peaks; _sh=round(p,4) in shoulder_qs
    ax[0].axvline(p,color=("r" if _am else "tab:cyan"),ls=(":" if _sh else ("-" if _am else "-.")),lw=1.3,alpha=0.9)
    _lab=(f"q={p:.2f}\nd={1/p:.2f}A" + ("\n(sh)" if _sh else "") + (f"\nnn={1.23/p:.2f}A" if p==halo_q else ""))
    ax[0].annotate(_lab,(p,Id[np.argmin(abs(qd-p))]),color=("r" if _am else "tab:cyan"),fontsize=7,ha="center",va="bottom")
import matplotlib.patches as mp
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[cc],label=cc) for cc in CANDIDATES]+
          [plt.Line2D([],[],color="r",label="amorphous halo (broad)"),plt.Line2D([],[],color="tab:cyan",ls="-.",label="crystalline ring (sharp)"),
           plt.Line2D([],[],color="k",ls=":",label="MAX detected ring")],fontsize=7,ncol=2)
ax[0].set_xlabel("q (1/A)"); ax[0].set_ylabel("I(q)"); ax[0].set_title("radial: detected halos (red, q&d) + candidate rings (dashed)")
# 오른쪽: 1D bandpass 신호 + 검출된 극대(=봉우리)를 세로선으로. ★봉우리는 bandpass 극대 그대로(가우시안 병합 없음)
_bpp=np.clip(bp1d,0,None)
ax[1].plot(qd[vsel],_bpp[vsel],"k-",lw=1.5,label="1D bandpass (=peak criterion)")
for p in halo_peaks:
    _am=p in amorph_peaks; _sh=round(p,4) in shoulder_qs; _yv=_bpp[int(np.argmin(np.abs(qd-p)))]
    ax[1].axvline(p,color=("r" if _am else "tab:cyan"),ls=(":" if _sh else ("-" if _am else "-.")),lw=1.2,alpha=0.9)
    ax[1].annotate(f"q={p:.3f}\nd={1/p:.2f}A"+("\n(sh)" if _sh else ""),(p,_yv),color=("r" if _am else "tab:cyan"),fontsize=7,ha="center",va="bottom")
ax[1].plot([],[],color="0.4",ls=":",lw=1.2,label="shoulder (2nd-deriv)")
ax[1].axhline(0,color="0.7",lw=0.6); ax[1].set_xlim(HALO_BG_BEAM,HALO_BG_QMAX)
ax[1].set_xlabel("q (1/A)"); ax[1].set_ylabel("bandpass"); ax[1].legend(fontsize=7)
ax[1].set_title("1D bandpass local maxima = halo peaks (red=amorph, cyan=cryst; add missed to HALO_QS_MANUAL)")
plt.tight_layout(); save(fig,"04_radial_expected"); plt.show()
# --- 그림 B: 각 halo를 2D bandpass NBD로 확인 — 평균 패턴에서 매끈한 radial 배경을 빼(=bandpass) 뿌연 링을 띄움 ---
H,W=meanpat.shape; _yy,_xx=np.mgrid[0:H,0:W]; _rr=np.hypot(_xx-center[0],_yy-center[1])*QPP
_bg1=np.interp(_rr.ravel(),qd,_u1d(Id,_ww)).reshape(H,W)         # 매끈한 radial 배경(2D)
resid2d=meanpat-_bg1                                             # 배경 뺀 잔차 = 링/halo가 뜸
nH0=len(halo_peaks); fig,ax=plt.subplots(1,nH0,figsize=(3.6*nH0,3.8),squeeze=False); ax=ax[0]
for a,p in zip(ax,halo_peaks):
    ann=(np.abs(_rr-p)<=BP_VIEWDQ)                              # 그 halo q-대역만
    if ann.sum()==0:                                           # q가 검출기 밖(반경 초과)
        a.text(0.5,0.5,f"q={p:.2f} off-detector",ha="center",va="center"); a.axis("off"); continue
    view=np.where(ann,resid2d,np.nan)
    vmax=np.nanpercentile(resid2d[ann],99)
    a.imshow(resid2d,cmap="magma",vmin=0,vmax=max(vmax,1),alpha=0.25)
    a.imshow(view,cmap="magma",vmin=0,vmax=max(vmax,1))
    a.add_patch(plt.Circle(center,p/QPP,fill=False,ec="cyan",ls="--",lw=1.0)); a.plot(*center,"c+",ms=7)
    a.set_title(f"bandpass @q={p:.2f} (d={1/p:.2f}A)\ndiffuse ring = real halo",fontsize=9); a.axis("off")
fig.suptitle("2D bandpass per detected halo: background-subtracted mean NBD, ring in the q-band = confirmed")
plt.tight_layout(); save(fig,"04_halo_bandpass"); plt.show()

## 4a) 켈리브레이션 **점검** — nn halo(≈2Å)로 현미경 q_per_px 검증. ★기본은 현미경 신뢰(보정 안 함)

**중요(질문 답):** I(q)의 **가장 센 봉우리 = RDF 1st peak(nn)** 가 **아님**. FSDP(최저 q, 가장 셈)는 **중거리질서(~4Å)**, **nn halo**(=RDF 1st peak, d≈2Å)는 **약한 고-q 봉우리**.
- ★그리고 "**2Å에 가장 가까운 봉우리를 2Å로 핀**"하는 규칙은 **항상 s_cal≈1을 만들어** 켈리를 *검증 못 하고 가정*함 → 함부로 핀하면 진짜 nn(예: 2.1Å)을 억지로 2.0으로 밀어 오히려 틀림.
- **현미경 q_per_px는 진짜 켈리**(카메라 길이·픽셀). nn halo가 **이미 1.9~2.3Å 범위**에 있으면 **켈리 맞음 → 보정 불필요**. (실제로 고차 halo d≈2.10/1.67/1.23Å이 LiF 2.01·Li2O 1.63·LiF 1.21 링과 보정 없이 일치.)
- 그래서 **기본 PIN_TO_2A=False = s_cal=1(현미경 신뢰)**. 2Å는 *점검용*. nn halo가 범위를 크게 벗어날 때만 PIN_TO_2A=True로 강제핀 검토. cepstral/RDF는 이 q범위서 안 되니 안 씀(불필요).

In [ ]:
# --- 이 셀 파라미터 ---
NN_TARGET_D  = 2.0      # nn 참고값(A) — '어느 halo가 nn인지' 식별·점검용(핀 목표 아님, 기본)
PIN_TO_2A    = False    # ★기본 False = 현미경 q_per_px 신뢰(s_cal=1). True = nn halo를 정확히 2A로 강제핀(켈리 오차 확증 시만)
NN_TOL_A     = 0.30     # nn halo가 2A ± 이 범위(A) 안이면 현미경 켈리 OK로 판단(1.7~2.3)
NN_ANCHOR_Q  = None     # nn halo q(1/A). None=자동: 공칭 d(=1/q)가 2A에 가장 가까운 봉우리
# FSDP(최저 q, 중거리 ~4A)는 nn 아님. nn halo = d~2A에 가장 가까운 봉우리(참고).
q_anchor = float(NN_ANCHOR_Q) if NN_ANCHOR_Q else min(halo_peaks,key=lambda p:abs(1.0/p-NN_TARGET_D))
nn_ok = abs(1.0/q_anchor-NN_TARGET_D)<=NN_TOL_A
q_cal_scale = ((1.0/NN_TARGET_D)/q_anchor) if PIN_TO_2A else 1.0    # 기본=보정없음(현미경 신뢰)
def qcal(p): return p*q_cal_scale
peaks_cal  = [qcal(p) for p in halo_peaks]
halo_q_cal = qcal(q_anchor)
print("=== 4a) 켈리브레이션 점검 (nn halo≈2A로 현미경 q_per_px 검증) ===")
print(f"  [halo 봉우리, 현미경 d]:")
for p in halo_peaks:
    tag=" <- nn halo(~2A, 점검용)" if abs(p-q_anchor)<1e-9 else (" (FSDP, 중거리)" if abs(p-halo_q)<1e-9 else "")
    print(f"    q={p:.3f}  d={1/p:.2f}A{tag}")
print(f"  nn halo(2A 최근접) = q{q_anchor:.3f} 현미경 d={1/q_anchor:.2f}A (2A와 {100*(1/q_anchor/NN_TARGET_D-1):+.0f}%)")
if nn_ok: print(f"  ✓ nn halo가 2A±{NN_TOL_A}A 범위 내 → 현미경 q_per_px 켈리 OK. 강제보정 불필요.")
else:     print(f"  ⚠ nn halo가 2A에서 크게 벗어남 → 실제 켈리 오차 가능. PIN_TO_2A=True 검토(또는 NN_ANCHOR_Q 확인).")
print(f"  → 적용 s_cal={q_cal_scale:.3f} ({'현미경 신뢰(보정 없음)' if not PIN_TO_2A else 'nn=2A 강제핀'}) | 사용 d(A)={[round(1/qcal(p),2) for p in halo_peaks]}")

## 4b) 비정질(amorphous) 구조 비교 — (§4a 보정) 검출 halo 봉우리 위치 ↔ 각 후보가 '비정질일 때' 예상 위치

각 후보가 **비정질**이라면 그 halo는 결정 링 위치(q=1/d) **근처(넓게)**에 뜬다. §4a로 **보정한 봉우리 위치**(빨간 선)와 각 후보 예상 위치(▽)를 겹쳐 본다.
- ★**중요(질문2 답)**: 후보 ▽ 위치는 **이론적 결정 d-spacing**이고, 세기는 **전자 구조인자**(Kirkland f_e, |F_hkl|²)로 계산 — **원자번호 효과**(Li 약산란)·소멸 반영(rough 공칭표 아님). **비정질은 저차 주halo만 강하게** 나오고 고-q(작은 d) 링은 감쇠.
  그래서 매칭은 **전자세기 상위 AM_KEEP개 링만**(진한 ▽) 씀. 고-q 약한 링(연한 ▽)은 **매칭서 제외**.
- **넓은 halo**라 tol 느슨. 판정은 **봉우리 위치**로. ※ **상 미확정**: 후보 nn거리 모두 ~2Å 축퇴 → 순위 참고만, confirm 없음. 물질 ID는 §4c/§6~§8.

In [ ]:
# --- 이 셀 파라미터 ---
AM_TOL  = 0.06    # 비정질 halo는 넓음 → 느슨한 위치 매칭 허용오차(1/A). (결정질 §4c는 tight)
AM_KEEP = 2       # ★비정질은 저차(강한) halo만 실측에 나옴 → 후보별 '전자세기' 상위 N개 링만 '비정질 예상 위치'로
# 후보별 '물리적으로 나올' 비정질 halo = 전자 구조인자 세기 상위 AM_KEEP개 링만. (ERINGS=전자세기, §1에서 계산)
_amrings={c:sorted(ERINGS[c],key=lambda t:-t[1])[:AM_KEEP] for c in CANDIDATES}
mrankA=fds.match_rings(peaks_cal,compounds=CANDIDATES,tol=AM_TOL,sigma=AM_TOL*0.6,rings=_amrings)
mscoreA={c:s for c,s,_,_,_ in mrankA}; mmatchA={c:mm for c,_,mm,_,_ in mrankA}
am_order=sorted(CANDIDATES,key=lambda c:-mscoreA[c])
print("=== 4b) 비정질 구조 비교 (전자세기 상위 저차 halo만, 상 미확정) ===")
print(f"  보정 halo 봉우리 d(A)={[round(1/p,2) for p in peaks_cal]} | s_cal={q_cal_scale:.3f} | tol={AM_TOL} | 후보 링=전자세기 상위 {AM_KEEP}개")
for c in am_order:
    kd=[round(d,2) for d,_ in _amrings[c]]
    print(f"  {c:7s} match={mscoreA[c]:.3f} matched_q={[round(x,3) for x in mmatchA[c]]} | 사용 링 d={kd} | nn거리={fds.COMPOUND_SHELLS[c][0][0]:.2f}A")
print(f"  ⚠ 후보 nn거리 모두 ~2A 축퇴 → 비정질만으론 상 구분 불가(순위 참고만). 물질 ID는 §4c/§6~§8.")
# 그림: 좌=보정 봉우리(빨간선) + 각 후보 예상 위치(진한 ▽=전자세기 상위 저차, 연한 ▽=약한 고-q) / 우=매칭 막대
fig,ax=plt.subplots(1,2,figsize=(15,4.4))
for p in peaks_cal: ax[0].axvline(p,color="tab:red",lw=2.0,alpha=0.85)
ax[0].annotate("nn halo (anchor)",(halo_q_cal,0.99),color="0.4",fontsize=7,ha="center",va="top")
ax[0].plot([],[],color="tab:red",lw=2,label="detected halo peak (calibrated)")
_yo={c:0.86-0.16*i for i,c in enumerate(am_order)}
for c in am_order:
    for dd,w in ERINGS[c]:
        qr=1/dd
        if 0.12<qr<1.1:
            phys=any(abs(dd-kd)<1e-9 and abs(w-kw)<1e-9 for kd,kw in _amrings[c])   # 전자세기 상위 저차 halo?
            on=phys and any(abs(qr-p)<=AM_TOL for p in peaks_cal)
            ax[0].plot([qr],[_yo[c]],marker="v",color=PHASE_COL[c],ms=(6+6*w) if phys else 5,
                       alpha=(0.95 if on else 0.6) if phys else 0.18,mec="k" if on else "none")
    ax[0].plot([],[],marker="v",color=PHASE_COL[c],ls="",label=f"{c} (nn{fds.COMPOUND_SHELLS[c][0][0]:.2f}A)")
ax[0].plot([],[],marker="v",color="0.5",ls="",alpha=0.18,label="weak/high-q ring (low electron |F|^2; excl.)")
ax[0].set_ylim(0,1.05); ax[0].set_xlim(0.12,1.1); ax[0].set_xlabel("q (1/A)  [calibrated x s_cal]"); ax[0].set_yticks([])
ax[0].set_title(f"amorphous: calib. peaks (red) vs electron-intensity rings (solid v = top {AM_KEEP})")
ax[0].legend(fontsize=6.5,loc="upper right")
xb=np.arange(len(am_order))
ax[1].bar(xb,[mscoreA[c] for c in am_order],color=[PHASE_COL[c] for c in am_order])
for i,c in enumerate(am_order): ax[1].text(i,mscoreA[c],f"nn{fds.COMPOUND_SHELLS[c][0][0]:.2f}",ha="center",va="bottom",fontsize=7)
ax[1].set_xticks(xb); ax[1].set_xticklabels(am_order,fontsize=8); ax[1].set_ylabel(f"match (halo vs top-{AM_KEEP} physical rings)")
ax[1].set_title("ranked by position match (NOT confirmed - amorphous nn~2A degenerate)")
plt.tight_layout(); save(fig,"04b_halo_amorphous"); plt.show()
save_csv("04b_halo_amorphous",["phase","match_score","matched_q","phys_rings_d","nn_dist_A"],
         [[c,f"{mscoreA[c]:.3f}",";".join(f"{x:.3f}" for x in mmatchA[c]),";".join(f"{d:.2f}" for d,_ in _amrings[c]),f"{fds.COMPOUND_SHELLS[c][0][0]:.2f}"] for c in am_order])

## 4c) 결정질(crystalline) 구조 비교 — (§4a 보정) 검출 봉우리 ↔ 후보 결정 링

§4a로 보정한 봉우리를 후보 5상 **결정 링**(전자세기)과 tight하게 대조. **cyan 선=보정 봉우리**, **▽=예상 링(크기=전자세기, solid=매칭)**.
- 켈리브레이션은 §4a에서 **물리 상수(nn≈2Å)** 로 이미 함(후보상으로 안 맞춤 → 순환논리 회피). 여기선 그 보정 q로 바로 대조.
- 링 세기는 **전자 구조인자**(원자번호·소멸 반영) — Li 약산란·소멸로 사라지는 링은 매칭 후보서 자동 제외.
- **score_phases 고유성**(한 상만 가진 링)만 `confirmed`. tight tol이라 넓은 halo가 우연히 걸치는 위양성 배제.
- ★넓은 halo는 본질적으로 결정 확정 근거가 약함 → 여기 확정은 **잠정**, 최종은 §6(MAX 날카로운 링)·§8(스팟).

In [ ]:
# --- 이 셀 파라미터 ---
FP_TOL      = 0.03    # 결정 링 매칭 허용오차(1/A) — 결정은 날카로우니 tight
FP_QCONFIRM = 0.30    # 이 q 이상 고유링만 '확정' 가능(저-q 배제)
EXCL_FSDP   = True    # FSDP(최저 q, 중거리질서 ~4A)는 결정 링 아님 → 결정 매칭서 제외. (nn 앵커·고차 halo는 포함)
# §4a 보정 봉우리를 결정링(ERINGS=전자세기)과 대조. nn 앵커는 최근접거리 링이라 포함, FSDP만 제외.
_fsdp_cal=qcal(halo_q)                                        # FSDP = 최저 q(중거리) — 결정 링 아님
peaks_c = [p for p in peaks_cal if (not EXCL_FSDP) or abs(p-_fsdp_cal)>1e-6]
def _mr(qs): return fds.match_rings(list(qs),compounds=CANDIDATES,tol=FP_TOL,sigma=FP_TOL*0.6,rings=ERINGS)
mrank=_mr(peaks_c); mscore={c:s for c,s,_,_,_ in mrank}; mmatch={c:mm for c,_,mm,_,_ in mrank}
eviD=fds.score_phases(peaks_c,peaks_c,candidates=CANDIDATES,tol=FP_TOL,q_confirm_min=FP_QCONFIRM,rings=ERINGS)
_vr={'confirmed':2,'possible':1,'weak/absent':0}
order=sorted(CANDIDATES,key=lambda c:(-_vr.get(eviD[c].verdict,0),-mscore.get(c,0)))
conf=[c for c in CANDIDATES if eviD[c].verdict=='confirmed']
poss=[c for c in CANDIDATES if eviD[c].verdict=='possible']
print("=== 4c) 결정질 구조 비교 (§4a 물리 보정 적용) ===")
print(f"  s_cal={q_cal_scale:.3f} | FSDP(q{qcal(halo_q):.3f},중거리) 제외 | 결정링 대상 보정 peak q={[round(p,3) for p in peaks_c]} d(A)={[round(1/p,2) for p in peaks_c]}")
for c in order:
    e=eviD[c]; print(f"  {c:7s} verdict={e.verdict:11s} unique_d={[round(x,2) for x in e.unique_d]} | match={mscore.get(c,0):.3f} matched_q={[round(x,3) for x in mmatch.get(c,[])]}")
print(f"  → 고유링 확정(잠정): {conf or '없음'} | 가능: {poss or '없음'} | 최종은 §6(MAX 날카로운 링)·§8(스팟)")
# 그림: 좌=보정 peak(cyan) vs 각 후보 결정링(▽) / 우=verdict 순위 막대
fig,ax=plt.subplots(1,2,figsize=(15,4.4))
for p in peaks_c: ax[0].axvline(p,color="tab:cyan",lw=2.0,alpha=0.85)
ax[0].plot([],[],color="tab:cyan",lw=2,label=f"detected peak (calibrated, s_cal={q_cal_scale:.3f})")
_yo={c:0.9-0.16*i for i,c in enumerate(order)}
for c in order:
    for dd,w in ERINGS[c]:
        qr=1/dd
        if 0.2<qr<1.15:
            on=any(abs(qr-p)<=FP_TOL for p in peaks_c)
            ax[0].plot([qr],[_yo[c]],marker="v",color=PHASE_COL[c],ms=6+5*w,alpha=1.0 if on else 0.3,mec="k" if on else "none")
    lbl=f"{c} [{eviD[c].verdict}]" if eviD[c].verdict!='weak/absent' else c
    ax[0].plot([],[],marker="v",color=PHASE_COL[c],ls="",label=lbl)
ax[0].set_ylim(0,1.05); ax[0].set_xlim(0.15,1.15); ax[0].set_xlabel("q (1/A)  [calibrated x s_cal]"); ax[0].set_yticks([])
ax[0].set_title("crystalline: calibrated peaks (cyan) vs candidate rings (v; solid=matched)"); ax[0].legend(fontsize=7,loc="upper right")
xb=np.arange(len(order))
ax[1].bar(xb,[mscore.get(c,0) for c in order],color=[PHASE_COL[c] for c in order],
          edgecolor=["k" if eviD[c].verdict=='confirmed' else "none" for c in order],linewidth=2)
for i,c in enumerate(order):
    if eviD[c].verdict=='confirmed': ax[1].text(i,mscore.get(c,0),"confirmed",ha="center",va="bottom",fontsize=8,color="k")
    elif eviD[c].verdict=='possible': ax[1].text(i,mscore.get(c,0),"possible",ha="center",va="bottom",fontsize=7,color="0.4")
ax[1].set_xticks(xb); ax[1].set_xticklabels(order,fontsize=8); ax[1].set_ylabel("tight ring position match")
ax[1].set_title("ranked by verdict then match (black edge = unique-ring CONFIRMED, provisional)")
plt.tight_layout(); save(fig,"04c_halo_crystalline"); plt.show()
save_csv("04c_halo_crystalline",["phase","match","matched_q","verdict","unique_d","cal_scale"],
         [[c,f"{mscore.get(c,0):.3f}",";".join(f"{x:.3f}" for x in mmatch.get(c,[])),eviD[c].verdict,";".join(f"{x:.2f}" for x in eviD[c].unique_d),f"{q_cal_scale:.3f}"] for c in order])

## 5) Halo 분석 — 물질/진공 판정 + 구조적 halo (진공 대비 cutoff)

**§4에서 검출한 실제 halo 봉우리**들을 프로브로 쓴다(고정 0.3/0.4/0.8 아님). 두 가지 맵:
- **PLAIN**(halo 밴드 세기): 물질/진공 판정용. **세기 ~ 두께**라 두꺼운 곳이 그냥 밝음(구조 아님).
- **STRUCTURAL**(halo *봉우리*): 각 픽셀의 radial을 **매끄러운 convex 배경**(빔꼬리+작은각; halo 밴드는 빼고 log-log
  power-law로 fit)으로 빼서 봉우리를 남긴다. ★직선 flank는 저-q 볼록 빔꼬리에서 **넘겨빼기(overshoot)** 해서 봉우리를 0으로
  죽였음(그래서 그래프가 나빴다) → **convex 배경**으로 교체. 게다가 봉우리 세기는 두께에 비례하므로,
  **contrast = 봉우리/배경**(두께 무관)을 진공 대비 z-score(≈4σ) → 두껍기만 한 featureless는 배제, **진짜 질서**만 밝음.
  넓은 halo도 통째로(낮고 높은 어깨까지) 배경 위로 올라오므로 자연히 잡힌다.
halo는 비정질이라 상 이름은 안 붙임(2Å 축퇴). 어떤 물질인지는 §6 ring/§7 spot이 답.

In [ ]:
# --- 이 셀 파라미터 ---
VAC_PCTL      = 15          # 엄격 기준진공 = 총세기 하위 X% (detector cutoff 기준). 물질이 화면을 많이 채우면 낮추기
HALO_BAND     = (0.15,0.60) # 물질 판정용 broad halo 밴드(1/A) — 이 구간 산란이 진공 대비 유의하면 물질
HALO_SIGMA    = 3.0         # 진공 대비 이 σ 이상 = 진짜 물질(cutoff). ★물질이 과하게 잡히면 키우기(4,5)
HALO_STRONG_S = 8.0         # 이 σ 이상 = 확정물질(강 halo), 그 사이는 예상물질
HALO_DQ       = 0.04        # 개별 halo 반경 PLAIN detector 반폭(1/A)
# STRUCTURAL: convex 배경(power-law) + contrast. §4 검출 봉우리를 프로브로 사용.
STRUCT_DQ     = 0.06        # halo 밴드 반폭(1/A) — 봉우리 폭에 맞춰(넓은 halo면 키우기)
STRUCT_BEAMCUT= 0.14        # 배경 fit 하한(1/A) — 이 아래 빔코어는 fit에서 제외
STRUCT_QMAX   = 1.05        # 배경 fit 상한(1/A)
STRUCT_DEG    = 2           # log-log 배경 다항 차수(2~3; power-law+상수 배경). 너무 크면 봉우리 흡수
STRUCT_CSIG   = 3.0         # ★질서 판정 σ 문턱: contrast(봉우리/배경)가 진공 대비 이 σ 이상인 픽셀만 '질서'로 셈(frac·tier용). 맵 색스케일과 무관
STRUCT_BGFLOOR= 0           # 배경 floor(진공 분위%). 기본 0=끔(robust MAD로 충분; 진공 배경이 0에 가까우면 floor가 오히려 csig 죽임)
NBIN          = 200
# 1) 픽셀 radial stack 1회 + 엄격 진공
q_stack,prof=fds.radial_stack(cube,center,QPP,q_max=1.2,nbin=NBIN,n_jobs=N_JOBS)
vac_ref=fds.strict_vacuum_mask(cube,center=center,q_per_px=QPP,pctl=VAC_PCTL)
# 2) 물질 판정 = broad halo 밴드 전체를 진공 대비(plain 환형; 정확한 peak 불필요, robust)
q0b=0.5*(HALO_BAND[0]+HALO_BAND[1]); dqb=0.5*(HALO_BAND[1]-HALO_BAND[0])
halo_sig,_=fds.detector_map(cube,center,QPP,q0b,dq=dqb,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))
material=halo_sig>HALO_SIGMA
halo_tier=np.where(~material,0,np.where(halo_sig>=HALO_STRONG_S,3,2))
print(f"물질 {100*material.mean():.0f}% (halo밴드 {HALO_BAND}>{HALO_SIGMA}σ) | 확정물질(>{HALO_STRONG_S}σ) {int((halo_tier==3).sum())}px | 예상물질 {int((halo_tier==2).sum())}px | 진공 {int((~material).sum())}px")
med_rad=prof[material.ravel()].mean(0) if material.any() else prof.mean(0)   # 물질 평균 radial
# 3) halo 프로브 = §4에서 검출한 실제 봉우리(고정값 아님). FSDP = 가장 강한 봉우리(§4 halo_q).
# §4 검출 봉우리 전체를 halo 맵 대상으로(폭분류는 표시용). 각 봉우리의 공간분포·NBD를 봄.
halo_probes=[round(p,3) for p in halo_peaks if STRUCT_BEAMCUT<p<STRUCT_QMAX-0.02]
if not halo_probes: halo_probes=[round(halo_q,3)]
fsdp_q=min(halo_probes,key=lambda p:abs(p-halo_q))            # 표시용 FSDP = 강한 봉우리에 가장 가까운 프로브
# PLAIN(세기=두께) & STRUCTURAL(contrast=봉우리/배경, convex 배경) 맵
plain_maps=[fds.detector_map(cube,center,QPP,p,dq=HALO_DQ,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))[0] for p in halo_probes]
_bump=fds.halo_bump_maps(cube,center,QPP,halo_probes,dq=STRUCT_DQ,beam_cut=STRUCT_BEAMCUT,q_max=STRUCT_QMAX,deg=STRUCT_DEG,vacuum_mask=vac_ref,bg_floor_pctl=STRUCT_BGFLOOR,_stack=(q_stack,prof))
struct_maps=[_bump[p]["csig"] for p in halo_probes]        # contrast 유의도(두께 무관)
struct_raw =[_bump[p]["raw"]  for p in halo_probes]         # 봉우리 세기(그림용)
# 대표로 보여줄 halo = STRUCTURAL이 가장 강한 halo(FSDP일 수도, 2·3차일 수도 — 데이터가 정함)
struct_frac=[float((sg>STRUCT_CSIG).mean()) for sg in struct_maps]
i_show=int(np.argmax(struct_frac)); q_show=halo_probes[i_show]; struct_sig=struct_maps[i_show]
print(f"검출 halo q ={[round(p,3) for p in halo_probes]}  d(A)={[round(1/p,2) for p in halo_probes]}  (FSDP={fsdp_q:.3f})")
print("  PLAIN  유의(>3σ) % (세기=두께):    ",[round(100*(sg>HALO_SIGMA).mean(),1) for sg in plain_maps])
print(f"  STRUCT 유의(>{STRUCT_CSIG}σ contrast) % (질서): ",[round(100*f,1) for f in struct_frac]," ← 두께무관, halo별로 다 봄")
print(f"  → STRUCTURAL 최강 halo = q{q_show:.3f} (d={1/q_show:.2f}A), {100*struct_frac[i_show]:.1f}%")
_sp=np.nanpercentile(struct_sig[material],[50,90,97,99]) if material.any() else [0,0,0,0]
print(f"  STRUCT csig(질서) 물질 분위 50/90/97/99% = {[round(x,1) for x in _sp]} — 상위만 크고 50%가 낮으면 질서가 국소적. 맵 vmax=97%로 그림(소수 outlier에 안 묻히게)")
# --- 그림 A: 대표 halo NBD(검출 halo 전부 원) | PLAIN | STRUCTURAL(최강 halo) | 물질 마스크 ---
iy,ix=np.unravel_index(int(np.argmax(np.where(material,halo_sig,-np.inf))),scan)
fig,ax=plt.subplots(1,4,figsize=(18,4.2))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
_hc=plt.cm.cool(np.linspace(0,1,len(halo_probes)))
for p,col in zip(halo_probes,_hc): ax[0].add_patch(plt.Circle(center,p/QPP,fill=False,ec=col,ls="--",lw=1.0))
ax[0].set_title(f"representative material pixel ({iy},{ix})\nall detected halos: {[round(p,2) for p in halo_probes]}",fontsize=9); ax[0].axis("off")
im=ax[1].imshow(np.where(material,halo_sig,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("PLAIN halo-band (intensity ~ thickness)"); ax[1].axis("off")
_svmax=float(np.nanpercentile(struct_sig[material],97)) if material.any() else 1.0
im=ax[2].imshow(np.where(material,struct_sig,np.nan),cmap="viridis",vmin=0,vmax=max(_svmax,1e-6)); plt.colorbar(im,ax=ax[2],fraction=0.046)
ax[2].set_title(f"STRUCTURAL: strongest halo q={q_show:.2f} (d={1/q_show:.2f}A)\n(contrast sig, vmax=97pct)",fontsize=9); ax[2].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmH=ListedColormap(["#000000","#4575b4","#2ca02c"]); nmH=BoundaryNorm([-.5,.5,2.5,3.5],3)
ax[3].imshow(halo_tier,cmap=cmH,norm=nmH); ax[3].set_title(f"material (>{HALO_SIGMA}s): vacuum/weak/strong"); ax[3].axis("off")
plt.tight_layout(); save(fig,"05_halo_material"); plt.show()
# --- 그림 B: 각 halo 반경 PLAIN(위) vs STRUCTURAL(아래) — 0.3/0.4/0.8이 진짜 봉우리인지 ---
nH=len(halo_probes)
fig,ax=plt.subplots(2,nH,figsize=(3.4*nH,7.0),squeeze=False)
for j,(p,pl,st) in enumerate(zip(halo_probes,plain_maps,struct_maps)):
    im=ax[0,j].imshow(np.where(material,pl,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[0,j],fraction=0.046)
    ax[0,j].set_title(f"PLAIN q={p:.2f} (d={1/p:.2f}A)",fontsize=9); ax[0,j].axis("off")
    _v=float(np.nanpercentile(st[material],97)) if material.any() else 1.0
    im=ax[1,j].imshow(np.where(material,st,np.nan),cmap="inferno",vmin=0,vmax=max(_v,1e-6)); plt.colorbar(im,ax=ax[1,j],fraction=0.046)
    ax[1,j].set_title(f"STRUCTURAL q={p:.2f} (contrast sig, vmax97%)",fontsize=9); ax[1,j].axis("off")
fig.suptitle("per-halo-radius: PLAIN (intensity/thickness) vs STRUCTURAL (contrast=bump/bg, thickness-free)")
plt.tight_layout(); save(fig,"05_halo_peaks"); plt.show()
# (convex 배경 + gaussian 분해 그림은 §4로 이동 — 여기선 맵만)
# --- 그림 C: 각 halo 반경에서 '강한 신호 픽셀'의 평균 NBD — 뿌연 링(halo)인가 스팟(결정질)인가 ---
STRONG_PCTL=90   # 각 반경 STRUCTURAL 맵 상위 %를 '강한 신호'로 골라 그 픽셀들의 NBD 평균
fig,ax=plt.subplots(1,nH,figsize=(3.6*nH,3.8)); ax=np.atleast_1d(ax)
for a,p,st in zip(ax,halo_probes,struct_maps):
    thr=np.nanpercentile(st[material],STRONG_PCTL) if material.any() else np.inf
    strong=material&(st>=thr)&(st>0)
    n=int(strong.sum())
    if n>=3:
        mp=fds.average_pattern(cube,strong)
        a.imshow((np.clip(mp,0,None)/np.max(mp))**0.3,cmap="magma"); a.plot(*center,"c+",ms=7)
        a.add_patch(plt.Circle(center,p/QPP,fill=False,ec="cyan",ls="--",lw=1.0))
        a.set_title(f"q={p:.2f} (d={1/p:.2f}A)\nmean NBD of {n} strong px",fontsize=9)
    else:
        a.set_title(f"q={p:.2f}: strong px<3",fontsize=9)
    a.axis("off")
fig.suptitle("mean NBD at STRONG STRUCTURAL pixels per halo radius: diffuse RING=amorphous halo, SPOTS=crystalline")
plt.tight_layout(); save(fig,"05_halo_meanNBD"); plt.show()
save_csv("05_halo",["halo_q","halo_d","plain_frac>3s",f"struct_frac>{STRUCT_CSIG}s(contrast)"],
         [[f"{p:.4f}",f"{1/p:.3f}",f"{(pl>HALO_SIGMA).mean():.4f}",f"{(st>STRUCT_CSIG).mean():.4f}"] for p,pl,st in zip(halo_probes,plain_maps,struct_maps)]
         +[["material_band","",f"{material.mean():.4f}",""]])

## 5-cep) Cepstral(EWPC) 맵 — 역공간(q)→실공간(d). structural이 못 본 halo를 보완

★**전에 "cepstral 안 된다"고 한 이유(정직)**: 그건 **평균 cepstrum의 radial 프로파일**이 (넓고 센 센터빔 지배로) **단조감소**라 깨끗한 nn 봉우리를 못 읽었다는 것 — **'거리 읽기(RDF)'용** 얘기였음. **여기 맵은 그것과 다른 것**을 봄:
- **per-pixel EWPC의 fluctuation(분산) 맵** `F=<C²>/<C>²−1`(FC-STEM 지표). 평균 cepstrum이 단조여도 **질서 있는 픽셀은 cepstral speckle(분산)이 커서** 매핑됨. EWPC는 **두께·빔에 강건**(log+power) → structural(halo contrast)이 SNR로 못 본 신호가 뜰 수 있음.
- **q↔d 매칭**: halo q0 → cepstral 거리 **r=1/q0=d0**. 각 halo d에서 fluctuation 맵 → §5 structural과 나란히 비교.
- 참고 그림: **평균 패턴 EWPC + radial**(nn 봉우리 보이나 확인). 안 보여도 fluctuation 맵은 별개.

In [ ]:
# --- 이 셀 파라미터 ---
CEP_OFFSET  = 1.0     # log(I+offset) — 0/저카운트 안정
CEP_WINDOW  = True    # Hann 창(가장자리 십자 아티팩트 억제)
CEP_RMIN    = 0.5     # radial 프로파일 중심 DC 제외(A)
_dpN=int(min(meanpat.shape)); CEP_DR=1.0/(_dpN*QPP)          # cepstral 해상도(A/px)
CEP_BANDHW  = max(0.35, 2.5*CEP_DR)                          # 각 halo d 밴드 반폭(A) — dr에 맞춰 자동
_rmax=(_dpN//2)*CEP_DR
print(f"=== 5-cep) Cepstral(EWPC) 맵 | dr={CEP_DR:.3f} A/px, 밴드반폭={CEP_BANDHW:.2f}A, quefrency 최대 {_rmax:.1f}A ===")
# 1) 참고: 평균 패턴 EWPC + radial (nn 봉우리 보이나 — 단조여도 아래 맵은 별개)
cep_disp=fds.ewpc_pattern(meanpat,offset=CEP_OFFSET,window=CEP_WINDOW)
r_ang,cep_prof=fds.cepstral_radial_profile(cep_disp,QPP,r_min=CEP_RMIN)
# 2) q->d: 각 halo 프로브 d에서 per-pixel fluctuation 맵
cep_ds=[1.0/p for p in halo_probes]
_bd=[(d,max(CEP_RMIN,d-CEP_BANDHW),min(d+CEP_BANDHW,_rmax-1e-3)) for d in cep_ds]
_bd=[(d,lo,hi) for d,lo,hi in _bd if lo<hi<_rmax]                # 유효 밴드만
cep_ds2=[d for d,_,_ in _bd]; bands=[(lo,hi) for _,lo,hi in _bd]
cep_maps=fds.fluctuation_multiband(cube,bands,QPP,offset=CEP_OFFSET,window=CEP_WINDOW,n_jobs=N_JOBS)
print("  halo d(A) 밴드 =",[round(d,2) for d in cep_ds2]," | 각 fluctuation 물질평균 =",[round(float(np.nanmean(cm[material])),3) for cm in cep_maps])
# --- 그림 A: 평균 EWPC(log) + radial 프로파일(halo d 표시) ---
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
_zt=int(min(_rmax/CEP_DR,60))
ax[0].imshow(np.log1p(cep_disp),cmap="magma")
_cc=(cep_disp.shape[1]//2,cep_disp.shape[0]//2); ax[0].plot(*_cc,"c+",ms=7)
ax[0].set_xlim(_cc[0]-_zt,_cc[0]+_zt); ax[0].set_ylim(_cc[1]+_zt,_cc[1]-_zt)
ax[0].set_title("mean-pattern EWPC (log) - real-space r"); ax[0].axis("off")
ax[1].plot(r_ang,cep_prof,"k-",lw=1.3)
for d in cep_ds2:
    ax[1].axvline(d,color="tab:red",ls="--",lw=1.0); ax[1].annotate(f"{d:.2f}A",(d,cep_prof.max()*0.9),fontsize=7,color="tab:red",ha="center",rotation=90)
ax[1].set_xlabel("quefrency r (A) = real-space distance"); ax[1].set_ylabel("cepstral intensity")
ax[1].set_title("EWPC radial (halo d marked; monotonic here = beam, but map below is variance)")
plt.tight_layout(); save(fig,"05cep_mean"); plt.show()
# --- 그림 B: 각 halo d에서 EWPC fluctuation 맵 (per-pixel; structural과 대응) ---
nC=len(cep_ds2)
fig,ax=plt.subplots(1,nC,figsize=(3.6*nC,3.9),squeeze=False); ax=ax[0]
for a,d,cm in zip(ax,cep_ds2,cep_maps):
    v=np.where(material,cm,np.nan); vmax=float(np.nanpercentile(v,97)) if material.any() else 1.0
    im=a.imshow(v,cmap="viridis",vmin=float(np.nanpercentile(v,5)) if material.any() else 0,vmax=max(vmax,1e-6)); plt.colorbar(im,ax=a,fraction=0.046)
    a.set_title(f"EWPC fluctuation @ r={d:.2f}A (q={1/d:.2f})\nvmax97%",fontsize=9); a.axis("off")
fig.suptitle("per-halo-distance EWPC fluctuation F=<C^2>/<C>^2-1 (bright=ordered speckle at that real-space distance)")
plt.tight_layout(); save(fig,"05cep_fluct"); plt.show()
save_csv("05cep",["halo_q","halo_d(=quefrency)","fluct_mean_material","fluct_p97"],
         [[f"{1/d:.4f}",f"{d:.3f}",f"{float(np.nanmean(cm[material])):.4f}",f"{float(np.nanpercentile(cm[material],97)):.4f}"] for d,cm in zip(cep_ds2,cep_maps)])

## 6-1) Ring 검출 + 물질확정 — MAX sharp 링 (§4 검출 + §4c 물질확정과 동일 형태)

§4가 halo를 (검출→§4c 확정)했듯, 여기선 **MAX NBD의 sharp 링**을 검출하고 후보 결정상과 매칭해 확정.
- **검출**: MAX radial에서 sharp 링(halo와 달리 날카로움; 소수 결정 픽셀도 MAX엔 또렷). FSDP·저-q 제외.
- **확정**: 검출 링 ↔ 후보 **전자링(ERINGS)** score_phases 고유성(§4c와 동일 로직). ★§4c는 넓은 halo라 근거 약(잠정) — **여기 sharp 링이 진짜 결정질 판정**.
- ★주의: 링 많은 상(Li2CO3 등)은 우연 고유링으로 위양성 가능 → **강한 링이 missing이면 의심**. 최종은 §7/§8 스팟이 재확인.

In [ ]:
# --- 이 셀 파라미터 ---
RING_QLO      = 0.24     # 링 검출 하한(1/A; FSDP 아래 제외)
RING_QHI      = 1.12     # 링 검출 상한
RING_BP_NARROW= 0.006    # ★bandpass 좁은 평활(1/A) — sharp 링용(작게=가까운 링도 분해). detect_rings보다 fine
RING_BP_BROAD = 0.05     # bandpass 넓은 평활(1/A) — 배경
RING_NSIG     = 1.2      # bandpass 봉우리 최소 prominence(노이즈 MAD 배수). ★잡링 많으면 키우기(1.5,2)
RING_TOPN     = 12       # 최대 검출 개수
RING_TOL      = 0.03     # 결정 링 매칭 tol(tight — sharp 링)
RING_QCONFIRM = 0.30     # 이 q 이상 고유링만 확정
RING_QS_MANUAL= None     # ★MAX에서 눈으로 본 링 직접 지정(예:[0.43,0.475,0.485]). None=자동
import matplotlib.patches as mp
from types import SimpleNamespace as _NS
from scipy.ndimage import uniform_filter1d as _u1r
from scipy.signal import find_peaks as _fpr
# 1) 검출: MAX radial 1D bandpass(§4와 동일 방식; sharp 링이라 좁은 창) — detect_rings와 달리 가까운 링(0.475/0.485)도 분해
def _par2(q,y,i):
    if 0<i<len(y)-1:
        den=y[i-1]-2*y[i]+y[i+1]
        if den!=0: return float(q[i]+0.5*(y[i-1]-y[i+1])/den*(q[i+1]-q[i]))
    return float(q[i])
_lIm=np.log(np.clip(Im,1e-3,None)); _dqm=float(np.median(np.diff(qm)))
_wn2=max(1,int(RING_BP_NARROW/_dqm)); _ww2=max(3,int(RING_BP_BROAD/_dqm))
bpm=_u1r(_lIm,_wn2)-_u1r(_lIm,_ww2)                          # MAX log I(q)의 DoG(sharp 링 부각)
_mw=(qm>RING_QLO)&(qm<RING_QHI); _nzm=1.4826*np.median(np.abs(bpm[_mw]-np.median(bpm[_mw])))+1e-9
if RING_QS_MANUAL:
    ring_qs=sorted(float(x) for x in RING_QS_MANUAL)
else:
    _pkr,_prr=_fpr(bpm,prominence=RING_NSIG*_nzm,distance=max(2,_wn2))   # distance 작게 → 가까운 링 분해
    _cand=sorted([(int(i),float(pr)) for i,pr in zip(_pkr,_prr['prominences']) if RING_QLO<qm[int(i)]<RING_QHI],key=lambda t:-t[1])[:RING_TOPN]
    ring_qs=sorted(_par2(qm,bpm,i) for i,_ in _cand)
print("=== 6-1) MAX sharp 링 검출(1D bandpass) + 물질확정 ===")
print(f"  검출 링 q={[round(r,3) for r in ring_qs]}  d(A)={[round(1/r,2) for r in ring_qs]}  (bandpass 창 narrow={_wn2*_dqm:.3f} distance={max(2,_wn2)}bin)")
# 2) 확정: 검출 링 ↔ 후보 전자링(score_phases 고유성; §4c와 동일)
_vr={'confirmed':2,'possible':1,'weak/absent':0}
if ring_qs:
    rmk=fds.match_rings(ring_qs,compounds=CANDIDATES,tol=RING_TOL,sigma=RING_TOL*0.6,rings=ERINGS)
    rms={c:s for c,s,_,_,_ in rmk}; rmm={c:mm for c,_,mm,_,_ in rmk}
    reviD=fds.score_phases(ring_qs,ring_qs,candidates=CANDIDATES,tol=RING_TOL,q_confirm_min=RING_QCONFIRM,rings=ERINGS)
else:
    rms={c:0.0 for c in CANDIDATES}; rmm={c:[] for c in CANDIDATES}
    reviD={c:_NS(verdict='weak/absent',unique_d=[],missing_strong_d=[]) for c in CANDIDATES}
rorder=sorted(CANDIDATES,key=lambda c:(-_vr[reviD[c].verdict],-rms.get(c,0)))
rconf=[c for c in CANDIDATES if reviD[c].verdict=='confirmed']; rposs=[c for c in CANDIDATES if reviD[c].verdict=='possible']
for c in rorder:
    e=reviD[c]; print(f"  {c:7s} verdict={e.verdict:11s} unique_d={[round(x,2) for x in e.unique_d]} missing_strong={[round(x,2) for x in getattr(e,'missing_strong_d',[])]} | match={rms.get(c,0):.3f} matched_q={[round(x,3) for x in rmm.get(c,[])]}")
print(f"  → sharp 링으로 확정: {rconf or '없음'} | 가능: {rposs or '없음'}"+("" if ring_qs else "  (MAX sharp 링 없음 = 결정질 링 근거 없음 → 비정질; §7/§8 재확인)"))
if rconf: print(f"  (확정 상의 강한 링이 missing이면 dense-ring 위양성 의심 — 위 missing_strong 확인)")
# 그림(§4c 형태): 좌=선택 peak↔후보 결정 peak(▽) / 중=1D bandpass(검출 기준) / 우=verdict 막대
fig,ax=plt.subplots(1,3,figsize=(19,4.4))
# ★§4c처럼: 검출(선택) 링을 세로선, 후보 전자링(=회절 peak, Z산란 반영)을 ▽(크기=세기, solid=매칭)로
for r in ring_qs: ax[0].axvline(r,color="k",lw=1.6,alpha=0.85)
ax[0].plot([],[],color="k",lw=1.6,label="selected peak (detected ring)")
_yo={c:0.9-0.16*i for i,c in enumerate(rorder)}
for c in rorder:
    for dd,w in ERINGS[c]:
        qr=1/dd
        if RING_QLO<qr<RING_QHI:
            on=any(abs(qr-r)<=RING_TOL for r in ring_qs)
            ax[0].plot([qr],[_yo[c]],marker="v",color=PHASE_COL[c],ms=6+6*w,alpha=1.0 if on else 0.3,mec="k" if on else "none")
    lbl=f"{c} [{reviD[c].verdict}]" if reviD[c].verdict!='weak/absent' else c
    ax[0].plot([],[],marker="v",color=PHASE_COL[c],ls="",label=lbl)
ax[0].set_ylim(0,1.05); ax[0].set_xlim(RING_QLO,RING_QHI); ax[0].set_yticks([]); ax[0].set_xlabel("q (1/A)")
ax[0].set_title("selected peaks (black) vs candidate electron rings (v; solid=matched, size=intensity)")
ax[0].legend(fontsize=6.5,loc="upper right")
# 중간: 1D bandpass(=검출 기준). 검출 링을 세로선으로 — §4처럼 가까운 링도 분해되는지 눈으로 확인
ax[1].plot(qm,bpm,"k-",lw=1.2,label="MAX 1D bandpass (=peak criterion)")
ax[1].axhline(RING_NSIG*_nzm,color="0.6",ls=":",lw=1.0,label=f"threshold {RING_NSIG}sigma")
for r in ring_qs: ax[1].axvline(r,color="tab:cyan",ls="-",lw=1.1,alpha=0.9); ax[1].annotate(f"{r:.3f}",(r,RING_NSIG*_nzm),fontsize=6,color="tab:cyan",ha="center",rotation=90,va="bottom")
ax[1].set_xlim(RING_QLO,RING_QHI); ax[1].set_xlabel("q (1/A)"); ax[1].set_ylabel("bandpass"); ax[1].legend(fontsize=7)
ax[1].set_title("1D bandpass local maxima = sharp rings (close rings resolved; add missed to RING_QS_MANUAL)")
xb=np.arange(len(rorder))
ax[2].bar(xb,[rms.get(c,0) for c in rorder],color=[PHASE_COL[c] for c in rorder],
          edgecolor=["k" if reviD[c].verdict=='confirmed' else "none" for c in rorder],linewidth=2)
for i,c in enumerate(rorder):
    if reviD[c].verdict=='confirmed': ax[2].text(i,rms.get(c,0),"confirmed",ha="center",va="bottom",fontsize=8)
    elif reviD[c].verdict=='possible': ax[2].text(i,rms.get(c,0),"possible",ha="center",va="bottom",fontsize=7,color="0.4")
ax[2].set_xticks(xb); ax[2].set_xticklabels(rorder,fontsize=8); ax[2].set_ylabel("tight ring match (sharp rings)")
ax[2].set_title("ranked by verdict then match (black edge = unique-ring CONFIRMED)")
plt.tight_layout(); save(fig,"06_1_ring_material"); plt.show()
save_csv("06_1_ring_material",["phase","match","matched_q","verdict","unique_d","missing_strong_d"],
         [[c,f"{rms.get(c,0):.3f}",";".join(f"{x:.3f}" for x in rmm.get(c,[])),reviD[c].verdict,";".join(f"{x:.2f}" for x in reviD[c].unique_d),";".join(f"{x:.2f}" for x in getattr(reviD[c],'missing_strong_d',[]))] for c in rorder])

## 6-3) Ring 공간맵 — §6-1 검출 링마다 **§5(halo)와 같은 형태** + 클러스터 검증 재확정

§5(halo) 흐름을 **§6-1에서 검출한 sharp 링**에 그대로 적용.
- ★**클러스터 검증(노이즈 배제)**: §6-1 bandpass는 노이즈도 좀 잡음. 하지만 **진짜 결정 링만 공간적으로 뭉친 클러스터**를 만듦(노이즈는 흩어짐). → **클러스터 있는 링만** 진짜로 보고 **그 링들로 상(相)을 재확정**(§6-1의 노이즈 포함 확정보다 깨끗).
- **그림 A (요약)**: **MAX NBD(결정질 픽셀들)** | PLAIN | STRUCTURAL | 결정영역 tier.
- **그림 B (PLAIN/STRUCT 격자)** — 아래줄 **STRUCTURAL = 각 링의 virtual image**(그 링이 어디서 뜨나) = ★**§7 흐름의 step 2**.
- **그림 C (강한 픽셀 MAX-NBD + 검출 스팟)**: 각 링 강한픽셀 **MAX 투영**(mean 아님)에 **잡힌 스팟(노란 원)** 표시 = ★**§7 흐름의 step 1(각 링에서 어떤 peak)**. 스팟=결정질, 뿌연 링=비정질.
- **그림 D (flank 검증)** — sharp 링=직선 flank 위 봉우리(bump>0).
※ **step 1·2(링별 스팟·virtual image)는 여기(§6-3)** 에서 봄. **step 3·4·5(영역 NBD-MAX→인덱싱→상맵)는 §7.**

In [ ]:
# --- 이 셀 파라미터 ---
RING_DQ       = 0.03   # 링 detector 반폭(1/A) ~ 링 폭
RING_FLANK    = 2.0    # flank 배수(sharp 링 배경빼기; halo와 달리 링엔 직선 flank가 맞다)
RING_SIGMA    = 4.0    # 비정질 배경 대비 이 σ 이상 = 진짜 sharp 링. ★잡링 많으면 키우기(5,6)
RING_MINCLUS  = 3      # 최소 cluster 크기(px) — 이보다 작은 점은 노이즈로 버림
from scipy.ndimage import label as _label
# 링 위치 = §6-1에서 검출·확정한 sharp 링(ring_qs) 그대로 사용.
print(f"=== 6-3) ring 공간맵: §6-1 검출 링 q(1/A) = {[round(r,3) for r in ring_qs]}  d(A) = {[round(1/r,2) for r in ring_qs]} ===")
def _flankbase(qq,p1d,q0,dq=RING_DQ,flank=RING_FLANK):        # peak_above_flank와 동일한 배경(양옆 flank 평균)
    band=(qq>=q0-dq)&(qq<=q0+dq); lo=(qq>=q0-flank*dq)&(qq<q0-dq); hi=(qq>q0+dq)&(qq<=q0+flank*dq)
    vals=[p1d[lo].mean() if lo.any() else np.nan,p1d[hi].mean() if hi.any() else np.nan]
    return band,lo,hi,float(np.nanmean(vals))
plain_rings=[]; struct_rings=[]; clusters=[]; cryst_region=np.zeros(scan,bool)
for rq in ring_qs:
    psig,_=fds.detector_map(cube,center,QPP,rq,dq=RING_DQ,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))   # PLAIN(세기)
    _,raw=fds.detector_map(cube,center,QPP,rq,dq=RING_DQ,flank=RING_FLANK,vacuum_mask=vac_ref,_stack=(q_stack,prof)) # flank 뺀 높이
    ssig=fds.significance(raw,material)                        # 비정질(물질) 배경 대비 z
    plain_rings.append(psig); struct_rings.append(ssig)
    lab,n=_label(material&(ssig>RING_SIGMA))
    sizes=[(g,int((lab==g).sum())) for g in range(1,n+1)]
    big=sorted([s for s in sizes if s[1]>=RING_MINCLUS],key=lambda s:-s[1])
    clusters.append([lab==g for g,_ in big])
    for m in clusters[-1]: cryst_region|=m
    print(f"  q={rq:.3f} (d={1/rq:.2f}A): STRUCT>{RING_SIGMA}s {int((material&(ssig>RING_SIGMA)).sum())}px, cluster {len(big)}개"+(f" (최대 {big[0][1]}px)" if big else ""))
ring_pred=cryst_region                                        # §7이 쓰는 결정질 후보 영역
print(f"  결정질(링) 영역 합집합 {int(cryst_region.sum())}px / 물질 {int(material.sum())}px")
nR=len(ring_qs)
# MAX-over-mask 헬퍼: 결정질 픽셀들의 NBD를 MAX 투영(sharp 스팟 보존; mean은 방위 다른 스팟을 링으로 뭉갬)
_flatp=cube._flat_patterns()
def _maxpat(mask):
    idx=np.flatnonzero(np.asarray(mask,bool).ravel())
    return _flatp[idx].max(0) if idx.size else np.zeros(meanpat.shape,float)
# ★클러스터 검증 재확정(Q2): §6-1은 검출 전부(노이즈 포함)로 확정→위양성. 여기선 '클러스터 있는 링'(공간적으로 뭉친 진짜 결정)만으로 재확정.
valid_rings=[rq for rq,cl in zip(ring_qs,clusters) if cl]
print(f"  ★클러스터 검증된 진짜 링(노이즈 제외) q={[round(r,3) for r in valid_rings]}  d(A)={[round(1/r,2) for r in valid_rings]}")
if valid_rings:
    _vms={c:s for c,s,_,_,_ in fds.match_rings(valid_rings,compounds=CANDIDATES,tol=RING_TOL,sigma=RING_TOL*0.6,rings=ERINGS)}
    _vevi=fds.score_phases(valid_rings,valid_rings,candidates=CANDIDATES,tol=RING_TOL,q_confirm_min=RING_QCONFIRM,rings=ERINGS)
    _vconf=[c for c in CANDIDATES if _vevi[c].verdict=='confirmed']
    for c in sorted(CANDIDATES,key=lambda c:-_vms.get(c,0)):
        e=_vevi[c]; print(f"    {c:7s} verdict={e.verdict:11s} unique_d={[round(x,2) for x in e.unique_d]} missing_strong={[round(x,2) for x in e.missing_strong_d]} match={_vms.get(c,0):.3f}")
    print(f"    → ★클러스터 검증 링으로 확정: {_vconf or '없음'} (노이즈 배제 후; 확정상 강한링 missing이면 여전히 의심)")
else:
    print("    (클러스터 있는 링 없음 → 공간적으로 뭉친 결정질 없음)")
from matplotlib.colors import ListedColormap,BoundaryNorm
# --- 그림 A: 대표 링픽셀 NBD | PLAIN(대표 링) | STRUCTURAL(대표 링) | 결정영역 — §5 05_halo_material와 동일 형태 ---
if nR>0:
    _si=int(np.argmax([float(np.nanmax(np.where(material,s,-np.inf))) for s in struct_rings]))  # 신호 최강 링
    rq0=ring_qs[_si]; st0=struct_rings[_si]
    iy2,ix2=np.unravel_index(int(np.argmax(np.where(material,st0,-np.inf))),scan)                # 대표 픽셀(최강 링신호)
    fig,ax=plt.subplots(1,4,figsize=(18,4.2))
    _repmask=cryst_region if cryst_region.any() else material           # 결정질 링 픽셀(없으면 물질)
    pat2=_maxpat(_repmask)                                              # ★MAX NBD(결정질 픽셀들) — sharp 링/스팟 보존
    ax[0].imshow((np.clip(pat2,0,None)/np.max(pat2))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
    _rc=plt.cm.cool(np.linspace(0,1,len(ring_qs)))
    for rq,col in zip(ring_qs,_rc): ax[0].add_patch(plt.Circle(center,rq/QPP,fill=False,ec=col,ls="--",lw=1.0))
    ax[0].set_title(f"MAX NBD of crystalline-ring pixels ({int(_repmask.sum())}px)\ndetected rings: {[round(r,3) for r in ring_qs]}",fontsize=8); ax[0].axis("off")
    im=ax[1].imshow(np.where(material,plain_rings[_si],np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[1],fraction=0.046)
    ax[1].set_title(f"PLAIN ring q={rq0:.3f} (intensity ~ thickness)",fontsize=9); ax[1].axis("off")
    _v0=float(np.nanpercentile(st0[material],97)) if material.any() else 1.0
    im=ax[2].imshow(np.where(material,st0,np.nan),cmap="viridis",vmin=0,vmax=max(_v0,1e-6)); plt.colorbar(im,ax=ax[2],fraction=0.046)
    ax[2].set_title(f"STRUCTURAL ring q={rq0:.3f}\n(sharp vs amorphous, vmax97%)",fontsize=9); ax[2].axis("off")
    _tier=np.where(~material,0,np.where(cryst_region,2,1))
    _cmR=ListedColormap(["#000000","#4575b4","#d62728"]); _nmR=BoundaryNorm([-.5,.5,1.5,2.5],3)
    ax[3].imshow(_tier,cmap=_cmR,norm=_nmR); ax[3].set_title("region: vacuum / amorphous material / crystalline-ring"); ax[3].axis("off")
    plt.tight_layout(); save(fig,"06_ring_material"); plt.show()
# --- §5(halo)와 동일 형태: 그림 B(PLAIN/STRUCT 격자) + C(강한픽셀 mean-NBD) + D(flank 검증) ---
nR=len(ring_qs)
RING_STRONG_PCTL=98    # 링은 sparse → STRUCTURAL 상위 % 픽셀을 '강한 신호'로(§5는 90; 링은 좁게)
if nR==0:
    print("  → MAX에서 halo와 구분되는 sharp 링 없음: §6 그림 생략, §7/§8(스팟·cepstral)에서 결정질 탐색")
else:
    # 그림 B: 각 링 반경 PLAIN(위) vs STRUCTURAL(아래) — §5 Fig B와 동일 형태
    fig,ax=plt.subplots(2,nR,figsize=(3.4*nR,7.0),squeeze=False)
    for j,(rq,pl,st) in enumerate(zip(ring_qs,plain_rings,struct_rings)):
        im=ax[0,j].imshow(np.where(material,pl,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[0,j],fraction=0.046)
        ax[0,j].set_title(f"PLAIN q={rq:.3f} (d={1/rq:.2f}A)\nintensity ~ thickness",fontsize=9); ax[0,j].axis("off")
        _v=float(np.nanpercentile(st[material],97)) if material.any() else 1.0
        im=ax[1,j].imshow(np.where(material,st,np.nan),cmap="inferno",vmin=0,vmax=max(_v,1e-6)); plt.colorbar(im,ax=ax[1,j],fraction=0.046)
        ax[1,j].set_title(f"STRUCTURAL q={rq:.3f} (sharp vs amorphous, vmax97%)",fontsize=9); ax[1,j].axis("off")
    fig.suptitle("per-ring-radius: PLAIN (intensity/thickness) vs STRUCTURAL (sharp ring vs amorphous background)")
    plt.tight_layout(); save(fig,"06_ring_peaks"); plt.show()
    # 그림 C: 각 링 반경 '강한 신호 픽셀'의 평균 NBD — sharp 스팟/링(결정질)? — §5 Fig C와 동일 형태
    fig,ax=plt.subplots(1,nR,figsize=(3.6*nR,3.9)); ax=np.atleast_1d(ax)
    for a,rq,st in zip(ax,ring_qs,struct_rings):
        thr=np.nanpercentile(st[material],RING_STRONG_PCTL) if material.any() else np.inf
        strong=material&(st>=thr)&(st>0); n=int(strong.sum())
        if n>=3:
            mpz=_maxpat(strong)                                # ★MAX 투영(sharp 스팟 보존; mean은 방위 스팟을 링으로 뭉갬)
            _sp=fds.detect_spots(mpz,center,QPP,n_mad=8.0,min_dist=3,tophat=11,q_max=1.15)   # ★그 링에서 잡힌 스팟(step1)
            a.imshow((np.clip(mpz,0,None)/np.max(mpz))**0.3,cmap="magma"); a.plot(*center,"c+",ms=7)
            a.scatter([s[0] for s in _sp],[s[1] for s in _sp],s=22,facecolors="none",edgecolors="yellow",lw=0.8)
            a.add_patch(plt.Circle(center,rq/QPP,fill=False,ec="cyan",ls="--",lw=1.0))
            a.set_title(f"q={rq:.3f} (d={1/rq:.2f}A)\nMAX NBD {n}px, {len(_sp)} spots",fontsize=8)
        else:
            a.set_title(f"q={rq:.3f}: strong px<3",fontsize=9)
        a.axis("off")
    fig.suptitle("per-ring detected SPOTS (yellow) on MAX NBD of strong pixels [step 1: which peaks per ring]")
    plt.tight_layout(); save(fig,"06_ring_meanNBD"); plt.show()
    # 그림 D: 각 링 flank 검증 — 링이 직선/평평 배경 위 봉우리인가(sharp 링=직선 flank; halo=convex)
    fig,ax=plt.subplots(1,nR,figsize=(3.6*nR,3.4)); ax=np.atleast_1d(ax)
    for a,rq,cl in zip(ax,ring_qs,clusters):
        reg_rad=prof[cl[0].ravel()].mean(0) if cl else med_rad
        band,lo,hi,base=_flankbase(q_stack,reg_rad,rq)
        win=(q_stack>=rq-3*RING_FLANK*RING_DQ)&(q_stack<=rq+3*RING_FLANK*RING_DQ)
        a.plot(q_stack[win],reg_rad[win],"k-",lw=1.3,label="region mean I(q)")
        for mask in (lo,hi):
            qs2=q_stack[mask]
            if qs2.size: a.axvspan(qs2.min(),qs2.max(),color="0.7",alpha=0.3)
        qb=q_stack[band]; a.plot([qb.min(),qb.max()],[base,base],"r--",lw=1.1,label="flank bg")
        a.fill_between(q_stack,base,reg_rad,where=band&(reg_rad>base),color="tab:orange",alpha=0.6,label="ring bump")
        a.axvline(rq,color="r",ls=":",lw=0.8)
        if q_stack[win].size: a.set_xlim(q_stack[win].min(),q_stack[win].max())
        bump=float(np.clip(reg_rad[band]-base,0,None).sum())
        a.set_title(f"flank q={rq:.3f}: bump={bump:.1f}\n({'real ring' if bump>0.5 else 'noise/flank'})",fontsize=8)
        a.set_xlabel("q (1/A)"); a.legend(fontsize=6)
    fig.suptitle("flank verification: a real sharp ring sits ABOVE a straight/flat local background (bump>0)")
    plt.tight_layout(); save(fig,"06_ring_flank"); plt.show()
save_csv("06_ring",["ring_q","ring_d","struct>sig_px","n_cluster","biggest_px","max_struct_sig"],
         [[f"{rq:.4f}",f"{1/rq:.3f}",int((material&(struct_rings[i]>RING_SIGMA)).sum()),len(clusters[i]),
           int(clusters[i][0].sum()) if clusters[i] else 0,f"{float(np.nanmax(struct_rings[i])):.1f}"] for i,rq in enumerate(ring_qs)])

## 7) 각 링 azimuthal unwrap → **spot 검출** (링당 6~10개)

★당신 설계. **모든 링, 모든 spot을 하나하나** 본다.
1. §6에서 찾은 **각 링 q**에서 **링을 직선으로 펼침**(azimuthal unwrap → 강도 vs 각도 1D).
2. 그 1D에서 **링 중앙값 위로 뚜렷이 솟은 봉우리 = spot**(링당 6~10). 매끈한 링=spot 0.
3. 링마다 [펼친 1D+spot | MAX 위 spot 위치] 표시. **각 spot의 virtual image는 §7b에서 spot마다.**

In [ ]:
# --- 파라미터 ---
RING_DQ       = 0.03   # 링 밴드 반폭(1/A)
RING_NSPOT    = 10     # ★링당 최대 spot 수(6~10)
RING_PROMFRAC = 0.06   # 펼친 1D 봉우리 최소 prominence(범위 대비)
RING_NSIG     = 4.0    # ★spot = 링 노이즈(MAD) 대비 이 σ 이상. 노이즈 적응(낮은 대비라도 이산이면 잡음; 매끈한 링=0). 놓치면 낮추기
RING_SEPDEG   = 12.0   # spot 최소 각도 간격(deg)
SPOTR         = 3      # circle detector 반경(px)
print("=== 7) 각 링 azimuthal unwrap → spot 검출 ===")
print(f"  §6 링 q(1/A) = {[round(q,3) for q in ring_qs]}")
ring_spots={}; spot_list=[]; nR=len(ring_qs)
if nR:
    fig,ax=plt.subplots(nR,2,figsize=(11,3.2*nR),squeeze=False)
    for i,rq in enumerate(ring_qs):
        sp,thd,prof=fds.ring_azimuthal_spots(mx,center,QPP,rq,dq=RING_DQ,max_spots=RING_NSPOT,min_prom_frac=RING_PROMFRAC,min_sep_deg=RING_SEPDEG,min_nsig=RING_NSIG)
        ring_spots[rq]=sp
        for j,(x,y,th,h) in enumerate(sp): spot_list.append((i+1,j+1,rq,x,y,th,h))
        ax[i,0].plot(thd,prof,"k-",lw=1.0); ax[i,0].axhline(np.median(prof),color="gray",ls=":",lw=0.8)
        for (x,y,th,h) in sp: ax[i,0].plot(np.degrees(th)%360,h,"rv",ms=7)
        ax[i,0].set_xlabel("azimuth (deg)"); ax[i,0].set_ylabel("ring intensity")
        ax[i,0].set_title(f"ring {i+1}: q={rq:.3f} (d={1/rq:.2f}A) unrolled: {len(sp)} spots",fontsize=9)
        ax[i,1].imshow((np.clip(mx,0,None)/max(np.max(mx),1e-9))**0.3,cmap="gray"); ax[i,1].plot(*center,"c+",ms=7)
        _tc=np.linspace(0,2*np.pi,180); ax[i,1].plot(center[0]+rq/QPP*np.cos(_tc),center[1]+rq/QPP*np.sin(_tc),"c-",lw=0.4,alpha=0.5)
        for j,(x,y,th,h) in enumerate(sp):
            ax[i,1].scatter([x],[y],s=45,facecolors="none",edgecolors="yellow",lw=1.1)
            ax[i,1].text(x+2,y+2,f"S{j+1}",color="yellow",fontsize=6)
        ax[i,1].set_title(f"ring {i+1}: {len(sp)} spots on MAX",fontsize=9); ax[i,1].axis("off")
    fig.suptitle("section 7: per-ring azimuthal unwrap -> spots")
    plt.tight_layout(); save(fig,"07_ring_spots"); plt.show()
    save_csv("07_ring_spots",["ring","ring_q","d_A","n_spots","spot_angles_deg"],[[i+1,round(rq,4),round(1/rq,2),len(ring_spots[rq]),";".join(f"{np.degrees(s[2])%360:.0f}" for s in ring_spots[rq])] for i,rq in enumerate(ring_qs)])
else:
    print("  §6 링 없음 — §7 생략")
print(f"  총 spot {len(spot_list)}개 (링 {nR}개) → §7b에서 spot마다 virtual image")

## 7b) **spot마다** virtual image → 강한 픽셀 → 클러스터

**spot 하나당 virtual image 하나.** (링3×spot6이면 virtual image 18개)
- spot에 circle detector → virtual image.
- **강한 픽셀**(물질 대비 >σ) → **뭉친 클러스터**(연결성분). image당 클러스터 몇 개가 나온다.
- 모든 virtual image를 격자로 다 보여준다. 라벨 `R{링}S{spot}`.

In [ ]:
# --- 파라미터 ---
VI_SIG      = 3.0    # ★virtual image 강한 픽셀: 물질 대비 이 σ 이상
CLUSTER_MIN = 3      # ★최소 클러스터 크기(px)
VI_NCOL     = 6      # 표시 격자 열 수
from scipy.ndimage import label as _label
_H,_W=meanpat.shape; _yy,_xx=np.mgrid[0:_H,0:_W]
print("=== 7b) spot마다 virtual image → 강한 픽셀 클러스터 ===")
spot_reg=[]   # {rid,sid,rq,vi,strong,labels,ncl}
for (rid,sid,rq,x,y,th,h) in spot_list:
    det=(np.hypot(_xx-x,_yy-y)<=SPOTR)
    vi=fds.virtual_image(cube,det)
    _vb=vi[material]; _vm=float(np.median(_vb)); _vd=1.4826*float(np.median(np.abs(_vb-_vm)))+1e-9
    strong=material&((vi-_vm)/_vd>VI_SIG)
    labels,ncl=_label(strong)
    # 최소크기 미만 클러스터 제거
    keep=np.zeros_like(labels)
    kk=0
    for g in range(1,ncl+1):
        if (labels==g).sum()>=CLUSTER_MIN: kk+=1; keep[labels==g]=kk
    spot_reg.append(dict(rid=rid,sid=sid,rq=rq,sx=x,sy=y,sth=th,vi=vi,strong=keep>0,labels=keep,ncl=kk))  # sx,sy=이 클러스터를 만든 링스팟 위치(§7c 고정 벡터용)
    print(f"  R{rid}S{sid} (q={rq:.3f}): 강한 {int((keep>0).sum())}px → 클러스터 {kk}개")
tot_cl=sum(r["ncl"] for r in spot_reg)
print(f"  => virtual image {len(spot_reg)}개, 클러스터 총 {tot_cl}개")
# 모든 virtual image 격자 표시(청크)
for c0 in range(0,len(spot_reg),VI_NCOL):
    grp=spot_reg[c0:c0+VI_NCOL]
    fig,ax=plt.subplots(1,len(grp),figsize=(2.7*len(grp),3.1),squeeze=False); ax=ax[0]
    for j,rec in enumerate(grp):
        _vv=np.where(material,rec["vi"],np.nan); _vx=float(np.nanpercentile(rec["vi"][material],99)) if material.any() else 1.0
        ax[j].imshow(_vv,cmap="inferno",vmax=max(_vx,1e-9))
        ax[j].contour(rec["labels"]>0,levels=[0.5],colors="cyan",linewidths=0.6)
        ax[j].set_title(f"R{rec['rid']}S{rec['sid']} q={rec['rq']:.3f}\n{rec['ncl']} clusters",fontsize=8); ax[j].axis("off")
    fig.suptitle(f"section 7b: per-spot virtual images ({c0+1}-{c0+len(grp)} of {len(spot_reg)})",fontsize=9)
    plt.tight_layout(); save(fig,f"07b_vimg_{c0//VI_NCOL:02d}"); plt.show()

## 7c) §7b 클러스터마다: log-스팟→역격자→실격자→cepstral 병진주기성(a,2a,3a) → indexing

★§7b에서 얻은 **클러스터**를 조사(전수 아님). 클러스터마다:
1. **Mean/MAX NBD** 생성(`CL_NBD`).
2. ★§6과 동일하게 **각 검출 링 q에서 azimuthal unwrap → 스팟(링당 0~`CL_NSPOT`개)**. 이러면 모든 스팟의 거리(q)가 링 거리와 자동 일치. **g1 = 이 클러스터를 만든 링스팟(무조건 고정)**, g2 = 각도 ≥`CL_MINANG`인 가장 가까운 스팟.
3. g → **실격자 벡터 a**(a·g=δ) — cepstral에서 spot 위치를 잡는 **조건**으로 쓴다.
4. **cepstral(NBD)** 에서 실격자 벡터로 중심→a→2a→3a 위치를 예측, 각 위치가 **벡터 방향 line profile에서 1차 미분=0 AND 2차 미분<0**(봉우리)에 맞으면 spot으로 지정(고립점 확인 위해 수직방향도 봉우리).
5. ★**a,2a,3a 연속 `CONSEC`개**가 spot이면 **결정**, 아니면 **비정질**.
6. 결정이면 스팟 g로 `index_pattern` → **상 확정**(≥4 자기일관)/예상.
7. 비정질이면 지배 거리 d를 후보 물질 **nn거리**(`COMPOUND_SHELLS`)에 매칭해 후보 표시.

특이: 스팟 2개(공선)면 벡터가 한 방향 → cepstral도 그 방향으로만 → **1D, 예상**으로 인덱싱.

In [ ]:
# --- 파라미터 (§7c: §7b 클러스터마다 log-스팟→역격자→실격자→cepstral 병진주기성) ---
CL_NBD        = "mean"  # 클러스터 NBD: "mean"(SNR↑) 또는 "max"(방위 다른 sharp 스팟 보존)
LOG_SPOT      = True    # azimuthal unwrap을 log(NBD)에서(약한 스팟↑)
LOG_OFF       = 1.0
CL_NSPOT      = 4       # ★§6처럼 각 링 azimuthal unwrap에서 링당 검출할 최대 스팟 수(0~4)
CL_RING_NSIG  = 3.0     # ★링 azimuthal 스팟 문턱(σ). 클러스터 평균 NBD는 약하니 §7(RING_NSIG=4)보다 낮게. 놓치면 2
NBD_SPOT_KEEP = 8       # 역격자/표시용 상위 스팟 수(링당 0~4 × 링수 → 상위 몇 개)
CL_MINANG     = 20      # ★두 역격자 벡터: g1=고정 링스팟, g2=각도 이 값(도) 이상인 가장 가까운 스팟
CONSEC        = 2       # ★cepstral 벡터방향 a→2a(→3a) 연속 봉우리 수 = 결정. 평균 NBD는 방위 섞여 3a가 약함 → 2(a·2a반복)로. 엄격히=3
CEP_TANGMIN   = 0.15    # ★봉우리 고립점 엄격도(수직곡률/1차곡률). 낮으면 병진 맞으면 관대하게 결정(예상)↑, 비정질 링/능선은 tangical≈0이라 여전히 배제. 더 관대=0.1, 엄격=0.4
CEP_SNAP      = 1.2     # ★cepstral 봉우리 판정 널럴하게: 예측 위치 ±이 값(A) 안의 가장 가까운 봉우리로 스냅. 1.2=45/60 결정(검증). 엄격히=0.7
CEP_SHOW_A    = 6.0     # cepstral 그림 표시 반경(A) — 모든 클러스터 동일 크기로
CEP_DR_TGT    = 0.10; LAT_TOL_G=0.06; LAT_HKMAX=2
IDX_TOLG      = 0.03; IDX_TOLANG=6.0; IDX_CONFIRM=4
AMORPH_TOL    = 0.35    # 비정질 지배 거리 d를 후보 nn거리에 매칭하는 허용오차(A)
CL_NCOL=6   # 그림 한 줄당 클러스터 수(모든 클러스터를 결정질 먼저로 다 표시)
_flatp=cube._flat_patterns(); _H,_W=meanpat.shape
_HPX=max(4,_H//32); _dr0=1.0/(_H*QPP); CEP_PAD=None if _dr0<=CEP_DR_TGT else int(2*round(0.5/(CEP_DR_TGT*QPP)))
print("=== 7c) §7b 클러스터마다: log-스팟→역격자→실격자→cepstral 병진주기성(a,2a,3a 연속) → index ===")
print(f"  cepstral pad={CEP_PAD} hp={_HPX}px | NBD={CL_NBD} | 결정=벡터방향 {CONSEC}연속 봉우리 | 클러스터 {sum(r['ncl'] for r in spot_reg)}개")
def _cl_nbd(m):
    idx=np.flatnonzero(np.asarray(m,bool).ravel())
    if idx.size==0: return np.zeros((_H,_W))
    blk=_flatp[idx]
    return blk.max(0).astype(np.float64) if CL_NBD=="max" else blk.mean(0,dtype=np.float64)
def _consec_run(orders_i):        # n=1(a)부터 연속으로 봉우리(applies)인 개수
    r=0
    for o in orders_i:
        if o[1]: r+=1
        else: break
    return r
def _dominant_d(nbd):             # cepstral 지배 거리 d(A) — 비정질 매칭용
    cep=fds.ewpc_pattern(nbd,pad=CEP_PAD,highpass=_HPX); rr,prof=fds.cepstral_radial_profile(cep,QPP,r_min=0.8)
    band=(rr>=0.8)&(rr<=6.0)
    return float(rr[band][int(np.argmax(prof[band]))]) if band.sum()>=2 else 0.0
def _amorph_match(d):             # d를 후보 nn거리에 매칭
    best=None; bd=AMORPH_TOL
    for ph in CANDIDATES:
        for sh in fds.COMPOUND_SHELLS[ph]:
            if abs(sh[0]-d)<bd: bd=abs(sh[0]-d); best=ph
    return best
spot_tier=np.zeros(scan,int); spot_phase=np.full(scan,-1,int); confirmed=set(); grain_rows=[]; cluster_recs=[]
for rec in spot_reg:
    for g in range(1,rec["ncl"]+1):
        m=rec["labels"]==g; nbd=_cl_nbd(m); lab=f"R{rec['rid']}S{rec['sid']}C{g}"; npx=int(m.sum())
        # 3) ★§6처럼 '각 링 q에서 azimuthal unwrap → 스팟(링당 0~CL_NSPOT개)'. 전 영역 2D검출 X → 거리(q)가 항상 링과 일치.
        det=np.log(np.clip(nbd,0,None)+LOG_OFF) if LOG_SPOT else nbd
        sp=[]
        for rq in ring_qs:
            _sr,_,_=fds.ring_azimuthal_spots(det,center,QPP,rq,dq=RING_DQ,max_spots=CL_NSPOT,min_prom_frac=RING_PROMFRAC,min_sep_deg=RING_SEPDEG,min_nsig=CL_RING_NSIG)
            for (x,y,th,h) in _sr: sp.append((x,y,rq))     # 링 스팟 → (x,y,q=rq)
        if len(sp)>NBD_SPOT_KEEP:                            # 너무 많으면 세기순 상위만
            _wall=[(float(det[int(np.clip(s[1],0,_H-1)),int(np.clip(s[0],0,_W-1))]),s) for s in sp]
            sp=[s for _,s in sorted(_wall,key=lambda t:-t[0])[:NBD_SPOT_KEEP]]
        # 4) ★g1 = 이 클러스터를 만든 링스팟(고정). g2 = 검출 스팟 중 g1과 각도 ≥ CL_MINANG인 중심 가까운 것.
        g1=np.asarray(fds.spots_to_gvectors([(rec["sx"],rec["sy"],rec["rq"])],center,QPP),float)[0]
        gv=np.asarray(fds.spots_to_gvectors(sp,center,QPP),float) if sp else np.zeros((0,2))
        lat=None; per=None; is1d=False; consec=0; g2=None
        if len(gv):
            for _j in np.argsort(np.hypot(gv[:,0],gv[:,1])):   # 중심에서 가까운 순
                if np.hypot(*(gv[_j]-g1))<1e-6: continue        # g1 자신 제외
                _cos=abs(float(g1@gv[_j])/(np.hypot(*g1)*np.hypot(*gv[_j])+1e-12))
                if np.degrees(np.arccos(np.clip(_cos,0,1)))>=CL_MINANG: g2=gv[_j]; break
        if g2 is not None:      # 2D: 링스팟 g1 + 각도 맞는 g2
            _grid=np.array([h*g1+k*g2 for h in range(-LAT_HKMAX,LAT_HKMAX+1) for k in range(-LAT_HKMAX,LAT_HKMAX+1) if not(h==0 and k==0)])
            lat={"g1":g1,"g2":g2,"grid_gs":_grid,"gvectors":gv,"lattice_frac":0.0}
            per=fds.cepstral_periodicity(nbd,g1,g2,QPP,pad=CEP_PAD,highpass=_HPX,tang_min=CEP_TANGMIN,snap_win=CEP_SNAP)
        else:                   # 각도 맞는 짝 없음 → 링스팟 벡터 하나로 1D. ★단, tangential 유지(비정질 링 배제: radial만이면 링도 통과함)
            is1d=True
            per=fds.cepstral_periodicity(nbd,g1,None,QPP,pad=CEP_PAD,highpass=_HPX,tang_min=CEP_TANGMIN,snap_win=CEP_SNAP)
        consec=max(_consec_run(per["orders"][i]) for i in per["orders"])
        cryst = consec>=CONSEC
        # 6-7) 결정→index(링스팟 g1 포함) / 비정질→d 매칭
        ph=None; conf=False; nm=0; zone="-"; dmatch=None; dd=0.0
        if cryst:
            gv_idx=np.vstack([g1[None,:],gv]) if len(gv) else g1[None,:]
            best,_=fds.index_pattern(gv_idx,candidates=CANDIDATES,tol_g=IDX_TOLG,tol_ang=IDX_TOLANG,min_spots=2,min_score=0.6,min_complete=0.5,confirm_min_spots=IDX_CONFIRM)
            if best is not None: ph=best["phase"]; conf=bool(best["indexed"]) and not is1d; nm=best["n_matched"]; zone=str(best["zone"])
            if ph is None:
                _ms=fds.match_rings([rec["rq"]]+[s[2] for s in sp],compounds=CANDIDATES,tol=IDX_TOLG*1.5,sigma=IDX_TOLG,rings=ERINGS)
                if _ms: ph=max(_ms,key=lambda t:t[1])[0]
        else:
            dd=_dominant_d(nbd); dmatch=_amorph_match(dd)
        cluster_recs.append(dict(lab=lab,m=m,nbd=nbd,sp=sp,lat=lat,per=per,cryst=cryst,consec=consec,is1d=is1d,ph=ph,conf=conf,zone=zone,nm=nm,dmatch=dmatch,dd=dd,g1=g1))
        if cryst and ph is not None:
            ki=CANDIDATES.index(ph)
            if conf: _up=m&(spot_tier<3); spot_tier[_up]=3; spot_phase[_up]=ki; confirmed.add(ph); tg="[확정]"
            else: _up=m&(spot_tier<2); spot_tier[_up]=2; spot_phase[_up]=ki; tg="[예상]"
            grain_rows.append([lab,npx,consec,len(sp),0 if is1d else 1,ph,int(conf),zone,nm])
            print(f"  {lab} {npx:3d}px: {consec}연속 스팟{len(sp)}{' 1D' if is1d else ''} → {tg} {ph:6s} zone{zone:12s} m{nm}")
        elif cryst:
            print(f"  {lab} {npx:3d}px: {consec}연속(결정) 스팟{len(sp)} → 상 미확정")
        else:
            print(f"  {lab} {npx:3d}px: {consec}연속(<{CONSEC}) 스팟{len(sp)} → 비정질"+(f" (d≈{dd:.2f}A ~ {dmatch})" if dmatch else ""))
_ncry=sum(1 for cr in cluster_recs if cr["cryst"])
print(f"  => 클러스터 {len(cluster_recs)}개 중 결정질 {_ncry}개 / 비정질 {len(cluster_recs)-_ncry}개 | 확정 상 {sorted(confirmed) or '없음'} | 확정 {int((spot_tier==3).sum())}px 예상 {int((spot_tier==2).sum())}px")
print(f"     [튜닝] 스팟 놓치면 CL_RING_NSIG↓(예:2) | cepstral 봉우리 놓치면 CEP_SNAP↑/CEP_TANGMIN↓ | 결정 놓치면 CONSEC↓ | 벡터 각도 CL_MINANG")
_cry=[cr for cr in cluster_recs if cr["cryst"]]; _amo=[cr for cr in cluster_recs if not cr["cryst"]]
_disp=_cry+_amo   # ★모든 클러스터 표시(결정질 먼저, 그 다음 비정질 전부)
print(f"  표시: 전체 {len(_disp)}개 (결정질 {len(_cry)}개 먼저 + 비정질 {len(_amo)}개)")
# ★모든 패널 동일 크기: cepstral 표시 반경을 고정(dr은 pad가 같아 전 클러스터 동일)
_drC=1.0/((CEP_PAD if CEP_PAD else _H)*QPP); _CN=(CEP_PAD if CEP_PAD else _H)
_CEPRR=int(min(_CN//2-2, round(CEP_SHOW_A/_drC)))
for c0 in range(0,len(_disp),CL_NCOL):
    grp=_disp[c0:c0+CL_NCOL]; fig,ax=plt.subplots(2,len(grp),figsize=(3.0*len(grp),6.0),squeeze=False)
    for j,cr in enumerate(grp):
        a=ax[0,j]; a.imshow((np.clip(cr["nbd"],0,None)/max(np.max(cr["nbd"]),1e-9))**0.3,cmap="gray"); a.plot(*center,"c+",ms=5)
        for rq in ring_qs: a.add_patch(plt.Circle(center,rq/QPP,fill=False,ec="cyan",ls="--",lw=0.5,alpha=0.55))  # ★§6 검출 링 — 모든 스팟은 이 원 위에 있어야
        for s in cr["sp"]: a.scatter([s[0]],[s[1]],s=42,facecolors="none",edgecolors="yellow",lw=0.9)
        if cr["lat"]:
            for gg,col in [(cr["lat"]["g1"],"yellow"),(cr["lat"]["g2"],"lime")]:
                a.arrow(center[0],center[1],gg[0]/QPP,gg[1]/QPP,color=col,width=0.5,head_width=2.5,length_includes_head=True,alpha=0.9)
        # ★고정된 링스팟 g1(이 클러스터를 만든 스팟) 위치 표시(빨강)
        _gg1=cr["lat"]["g1"] if cr["lat"] else (cr.get("g1"))
        if _gg1 is not None: a.scatter([center[0]+_gg1[0]/QPP],[center[1]+_gg1[1]/QPP],s=90,facecolors="none",edgecolors="red",lw=1.4)
        _res=(cr["ph"]+(" [C] zone"+cr["zone"] if cr["conf"] else " [P]")) if cr["ph"] else (("amorph~"+str(cr["dmatch"])) if cr["dmatch"] else ("amorphous" if not cr["cryst"] else "unindexed"))
        a.set_xlim(0,_W); a.set_ylim(_H,0)                    # ★NBD 패널 크기 고정(오버레이가 축을 안 늘리게)
        a.set_title(f"{cr['lab']} {len(cr['sp'])}sp {cr['consec']}consec -> {_res}",fontsize=7); a.axis("off")
        a2=ax[1,j]; per=cr["per"]
        cep=fds.ewpc_pattern(cr["nbd"],pad=CEP_PAD,highpass=_HPX); _n=cep.shape[0]; _cc=_n//2
        _rr=min(_cc-2,_CEPRR)                                 # ★고정 반경 → 모든 cepstral 동일 크기
        a2.imshow(cep[_cc-_rr:_cc+_rr,_cc-_rr:_cc+_rr]**0.5,cmap="magma"); a2.plot(_rr,_rr,"w+",ms=6)
        if per is not None:
            for i,av in enumerate(per["a_vectors"]):
                # ★cepstral에 실공간 벡터(파랑) 화살표
                a2.arrow(_rr,_rr,av[0]/per["dr"],av[1]/per["dr"],color="deepskyblue",width=0.4,head_width=2.2,length_includes_head=True,alpha=0.95,zorder=6)
                for ni,od in enumerate(per["orders"][i]):
                    n=ni+1; px=_rr+n*av[0]/per["dr"]; py=_rr+n*av[1]/per["dr"]
                    if 0<=px<2*_rr and 0<=py<2*_rr:
                        ok=bool(od[1]); a2.scatter([px],[py],s=70,facecolors=("lime" if ok else "none"),edgecolors=("lime" if ok else "red"),lw=1.3,zorder=5)
        a2.set_xlim(0,2*_rr); a2.set_ylim(2*_rr,0)            # ★cepstral 패널 크기 고정(화살표가 축을 안 늘리게)
        a2.set_title(f"cepstral realspace vec (blue): {cr['consec']}consec ({'CRYSTAL' if cr['cryst'] else 'amorph'})",fontsize=7); a2.axis("off")
    fig.suptitle(f"section 7c [clusters, crystalline first] ({c0+1}-{c0+len(grp)}, {_ncry} crystalline of {len(cluster_recs)}): (top) {CL_NBD} NBD+log-spots+reciprocal lattice | (bottom) cepstral a,2a,3a green=peak red=none",fontsize=8)
    plt.tight_layout(); save(fig,f"07c_cluster_{c0//CL_NCOL:02d}"); plt.show()
from matplotlib.colors import ListedColormap,BoundaryNorm
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
fig,ax=plt.subplots(1,1,figsize=(6,4.4))
ax.imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
ax.imshow(np.where(spot_tier==3,spot_phase,np.nan),cmap=cmP,norm=nmP)
ax.imshow(np.where(spot_tier==2,spot_phase,np.nan),cmap=cmP,norm=nmP,alpha=0.45)
ax.set_title("section 7 result: solid=confirmed (zone-indexed), faded=predict"); ax.axis("off")
ax.legend(handles=[plt.Line2D([],[],marker="s",ls="",color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="upper right")
plt.tight_layout(); save(fig,"07_phasemap"); plt.show()
if grain_rows: save_csv("07_clusters",["cluster","size","consec","n_spots","is_2d","phase","confirmed","zone","n_matched"],grain_rows)

## 8) Phase mapping — 확정 / 예상 / 결정질(상불명) / 비정질 (종합)

상 이름은 **spot 인덱싱(§7)만** 붙인다(신호 우선 원칙): **확정**=≥4스팟 자기일관 · **예상**=2스팟.
**결정질(상불명)**=§6 링 신호는 진짜인데 spot 인덱싱이 상을 확정 못 한 영역. **비정질**=halo만(§5). **없음**=진공.
확정 > 예상 > 결정질 > 비정질 우선.

In [ ]:
# 종합 tier: 4 확정(spot) > 3 예상(spot) > 2 결정질-상불명(ring) > 1 비정질(halo) > 0 진공
tier=np.where(~material,0,1)                              # 물질=비정질 기본
phase_idx=np.full(scan,-1,int)
cn=cryst_region&material                                  # §6 링 신호 = 진짜 결정질(상 이름은 아직)
tier[cn]=2
pm=(spot_tier==2)                                         # §7 spot 예상(2스팟) → 상 이름 있음
tier[pm]=np.maximum(tier[pm],3); phase_idx[pm]=spot_phase[pm]
cm=(spot_tier==3)                                         # §7 spot 확정(≥4스팟) — 최우선
tier[cm]=4; phase_idx[cm]=spot_phase[cm]
print("=== 픽셀 종합 판정 ===")
for t,name in [(4,"확정"),(3,"예상"),(2,"결정질(상불명)"),(1,"비정질"),(0,"없음")]: print(f"  {name}: {int((tier==t).sum())} px | ",end="")
print()
for k,c in enumerate(CANDIDATES):
    cf=int(((phase_idx==k)&(tier==4)).sum()); pr=int(((phase_idx==k)&(tier==3)).sum())
    if cf or pr: print(f"    {c:7s}: 확정 {cf} px, 예상 {pr} px")
conf_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((phase_idx==k)&(tier==4)).any()})
pred_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((phase_idx==k)&(tier==3)).any()})
print(f"  => 확정 상: {conf_ph or '없음'} | 예상 상: {pred_ph or '없음'} | 결정질(상불명) {int((tier==2).sum())}px")
from matplotlib.colors import ListedColormap,BoundaryNorm
import matplotlib.patches as mp
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
def layer(t): return np.where((tier==t)&(phase_idx>=0),phase_idx,np.nan)
fig,ax=plt.subplots(1,4,figsize=(19,4.4))
ax[0].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[0].imshow(layer(4),cmap=cmP,norm=nmP)
ax[0].set_title(f"CONFIRMED (spot indexed): {int((tier==4).sum())} px"); ax[0].axis("off")
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
ax[1].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[1].imshow(layer(3),cmap=cmP,norm=nmP)
ax[1].set_title(f"PREDICTED (2-spot): {int((tier==3).sum())} px"); ax[1].axis("off")
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[2].imshow(np.where(tier==2,1.0,np.nan),cmap="cool",vmin=0,vmax=1.4)
ax[2].set_title(f"CRYSTALLINE, phase unknown (ring): {int((tier==2).sum())} px"); ax[2].axis("off")
ax[3].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[3].imshow(np.where(tier==1,1.0,np.nan),cmap="Greys",vmin=0,vmax=1.4)
ax[3].set_title(f"AMORPHOUS (halo only): {int((tier==1).sum())} px"); ax[3].axis("off")
fig.suptitle("PHASE MAP - CONFIRMED / PREDICTED (phase named) | CRYSTALLINE-unknown / AMORPHOUS")
plt.tight_layout(); save(fig,"08_phase_map"); plt.show()
save_csv("08_phase_tiers",["phase","confirmed_px","predicted_px"],
         [[c,int(((phase_idx==CANDIDATES.index(c))&(tier==4)).sum()),int(((phase_idx==CANDIDATES.index(c))&(tier==3)).sum())] for c in CANDIDATES]
         +[["_crystalline_unknown","",int((tier==2).sum())],["_amorphous","",int((tier==1).sum())]])

## 8b) 비정질/나노결정 상 — FC-STEM 구조선택 NBD → §4 봉우리 → Debye(비정질+결정질)

§5 FC-STEM fluctuation으로 거리밴드별 질서 픽셀을 골라 **Mean NBD** 를 만들고(baseline 차감 없음), 4개 셀로 나눠 진행한다.
- **8b-1 (STEP1+2)** 강한 fluctuation 픽셀 → Mean NBD → **§4식 convex+bandpass 봉우리**(nn~2Å 포함, 회색 표시).
- **8b-2 (STEP3, 참고)** 각 링의 **radial FWHM**(좁=나노결정 링 / 넓=비정질 halo)과 **방위각 V**(spotty=소수결정) — 방위각 프로파일도 표시. am↔nano는 연속(참고용).
- **8b-3 (STEP4a)** 검출 링을 **§4와 동일한 Debye 매칭 — 비정질(§4b: 느슨·전자세기 상위 저차) / 결정질(§4c: tight·전체 링) 둘 다** 좌/우로. nn~2Å는 표시하되 **결정 구별은 nn 외 링**이 함.
- **8b-4 (STEP4b)** 결정링 확정 픽셀만 상 매핑 + §7 결정 grain 합성 + CSV.

In [ ]:
# === 8b-1) STEP1+2: FC-STEM 구조선택 Mean NBD → §4 radial 봉우리 ===
import matplotlib.patches as mp
FC_STRONG_PCTL=90; FC_MINPX=15
BG_DQ=0.06; BG_BEAM=0.14; BG_QMAX=min(1.05,RING_QHI); PEAK_PROMF=0.05   # §4식 halo-excising 배경(convex 흡수 방지)
NN=(0.42,0.58)                    # nn ~2Å 공통 밴드(모든 후보) — 표시하되 구별엔 약함
from scipy.signal import find_peaks
_HQ=sorted(set(round(1.0/float(d),3) for d in cep_ds2))   # 알려진 halo 위치(=§4/§5 밴드 거리들) → 배경 fit서 제외
_amask=material&(tier<=1); _cx,_cy=float(center[0]),float(center[1])
band_recs=[]
for i,d in enumerate(cep_ds2):
    fmap=np.asarray(cep_maps[i],float)
    thr=float(np.nanpercentile(fmap[_amask],FC_STRONG_PCTL)) if _amask.any() else np.inf
    strong=_amask&(fmap>=thr)
    rec=dict(d=float(d),strong=strong,nbd=None,q=None,sig=None,rings=[],ph=None)
    if int(strong.sum())>=FC_MINPX:
        nbd=fds.average_pattern(cube,strong); q,I=fds.azimuthal_integrate(nbd,center,QPP)
        bg=fds.halo_background_1d(q,I,_HQ,dq=BG_DQ,beam_cut=BG_BEAM,q_max=BG_QMAX,deg=2)   # ★알려진 halo 제외하고 배경 fit → 흡수 방지(§4 방식)
        m=(q>=RING_QLO)&(q<=RING_QHI); sig=np.zeros_like(q,float)
        sig[m]=np.clip(I-bg,0,None)[m]/np.clip(bg,1e-6,None)[m]                            # contrast=(I-bg)/bg
        _pi,_=find_peaks(sig,prominence=PEAK_PROMF*(float(sig[m].max())+1e-9),distance=max(2,int(0.03/max(q[1]-q[0],1e-6))))
        pk=[float(q[j]) for j in _pi if RING_QLO<q[j]<RING_QHI]
        rings=[dict(q=float(pp),d=1.0/float(pp),shared=bool(NN[0]<=pp<=NN[1])) for pp in pk]
        rec.update(nbd=nbd,q=q[m],sig=sig[m],rings=rings)
    band_recs.append(rec)
print("=== 8b-1) 구조선택 NBD 봉우리 (nn~2Å 포함) ===")
for br in band_recs:
    if br["nbd"] is None: print(f"  r={br['d']:.2f}A: 픽셀부족"); continue
    print(f"  r={br['d']:.2f}A strong{int(br['strong'].sum()):4d}px 링 q={[round(r['q'],3) for r in br['rings']]} (nn공통 {[round(r['q'],3) for r in br['rings'] if r['shared']]})")
_nb=len(band_recs); fig,ax=plt.subplots(_nb,3,figsize=(12,2.7*_nb),squeeze=False)
for i,br in enumerate(band_recs):
    a=ax[i,0]; a.imshow(np.where(material,0.1,np.nan),cmap="Greys",vmin=0,vmax=1); a.imshow(np.where(br["strong"],1.0,np.nan),cmap="autumn",vmin=0,vmax=1)
    a.set_title(f"STEP1 r={br['d']:.2f}A strong {int(br['strong'].sum())}px",fontsize=8); a.axis("off")
    a1=ax[i,1]
    if br["nbd"] is not None:
        a1.imshow((np.clip(br["nbd"],0,None)/max(np.max(br["nbd"]),1e-9))**0.3,cmap="gray"); a1.plot(_cx,_cy,"c+",ms=4)
        for r in br["rings"]: a1.add_patch(plt.Circle((_cx,_cy),r["q"]/QPP,fill=False,ec=("0.6" if r["shared"] else "cyan"),ls="--",lw=0.7,alpha=0.85))
    a1.set_title("mean NBD (grey=nn~2A shared, cyan=other)",fontsize=8); a1.set_xlim(0,_W); a1.set_ylim(_H,0); a1.axis("off")
    a2=ax[i,2]
    if br["sig"] is not None:
        a2.plot(br["q"],br["sig"],"k-",lw=1.1); a2.axvspan(*NN,color="0.9",alpha=0.7)
        for r in br["rings"]: a2.axvline(r["q"],color=("0.6" if r["shared"] else "tab:red"),lw=1.2)
    a2.set_title("STEP2 halo-excising contrast (§4) peaks (grey band=nn~2A)",fontsize=8); a2.set_xlabel("q (1/A)")
fig.suptitle("8b-1) per-distance | strong px + mean NBD | §4 radial peaks (nn~2A shown in grey)",fontsize=8)
plt.tight_layout(); save(fig,"08b_fcstem_bands"); plt.show()

In [ ]:
# === 8b-2) STEP3: 링 성격 — radial FWHM + 방위각 V (참고) ===
from scipy.signal import peak_widths
from scipy.ndimage import map_coordinates
FWHM_SHARP=0.08; AZ_NBAND=3; _cx,_cy=float(center[0]),float(center[1])
def _radial_fwhm(q,sig,qp):
    i=int(np.argmin(np.abs(q-qp)))
    try: return float(peak_widths(sig,[i],rel_height=0.5)[0][0]*float(np.median(np.diff(q))))
    except Exception: return float("nan")
def _azim_profile(nbd,qp):
    rpx=qp/QPP; ang=np.linspace(0,2*np.pi,180,endpoint=False); acc=[]
    for drr in range(-AZ_NBAND,AZ_NBAND+1):
        ys=_cy+(rpx+drr)*np.sin(ang); xs=_cx+(rpx+drr)*np.cos(ang)
        acc.append(map_coordinates(np.asarray(nbd,float),[ys,xs],order=1,mode="nearest"))
    prof=np.mean(acc,axis=0); return float(np.var(prof)/(np.mean(prof)**2+1e-12)),np.degrees(ang),prof
_allr=[]
for br in band_recs:
    if br["nbd"] is None: continue
    for r in br["rings"]:
        r["fwhm"]=_radial_fwhm(br["q"],br["sig"],r["q"]); V,ang,prof=_azim_profile(br["nbd"],r["q"])
        r["V"]=V; r["char"]=("nano-ring" if (r["fwhm"]==r["fwhm"] and r["fwhm"]<=FWHM_SHARP) else "amorph-halo")
        _allr.append((br["d"],r,ang,prof))
print("=== 8b-2) 링 성격 (FWHM 좁=nano-ring, 넓=amorph-halo; V 큼=spotty) ===")
for d,r,_,_ in _allr:
    print(f"  r={d:.2f}A q={r['q']:.3f} d={r['d']:.2f}A FWHM={r['fwhm']:.3f} V={r['V']:.2f} -> {r['char']}"+(" (nn~2A 공통)" if r["shared"] else ""))
if _allr:
    _n=len(_allr); _c=min(4,_n); _rw=int(np.ceil(_n/_c)); fig,ax=plt.subplots(_rw,_c,figsize=(3.3*_c,2.4*_rw),squeeze=False)
    for k,(d,r,ang,prof) in enumerate(_allr):
        a=ax[k//_c,k%_c]; a.plot(ang,prof/(prof.mean()+1e-9),"k-",lw=1.0); a.axhline(1,color="0.6",lw=0.5)
        a.set_title(f"r={d:.2f} q={r['q']:.2f} {r['char']} FWHM={r['fwhm']:.3f} V={r['V']:.2f}",fontsize=7); a.set_xlabel("azimuth (deg)",fontsize=7)
    for k in range(_n,_rw*_c): ax[k//_c,k%_c].axis("off")
    fig.suptitle("8b-2) per-ring azimuthal I(theta): flat=halo(low V), spiky=nano spots(high V); radial FWHM=sharpness",fontsize=9)
    plt.tight_layout(); save(fig,"08b_ring_character"); plt.show()

In [ ]:
# === 8b-3) STEP4a: Debye 매칭 — 비정질(§4b) · 결정질(§4c) 둘 다 (§4 동일 기준) ===
NN=(0.42,0.58); AM_TOL=0.06; AM_KEEP=2; FP_TOL=0.03; FP_QCONFIRM=0.30
_amrings={c:sorted(ERINGS[c],key=lambda t:-t[1])[:AM_KEEP] for c in CANDIDATES}   # §4b: 전자세기 상위 저차
PKS=sorted(set(round(r["q"],3) for br in band_recs for r in br["rings"]))          # 모든 링(nn 포함)
PKC=sorted(set(round(r["q"],3) for br in band_recs for r in br["rings"] if not r["shared"]))  # 결정 구별용(nn 외)
mrankA=fds.match_rings(PKS,compounds=CANDIDATES,tol=AM_TOL,sigma=AM_TOL*0.6,rings=_amrings) if PKS else []
mrankC=fds.match_rings(PKS,compounds=CANDIDATES,tol=FP_TOL,sigma=FP_TOL*0.6,rings=ERINGS) if PKS else []
evi=fds.score_phases(PKC,PKC,candidates=CANDIDATES,tol=FP_TOL,q_confirm_min=FP_QCONFIRM,rings=ERINGS) if PKC else {}
scA={c:s for c,s,_,_,_ in mrankA}; scC={c:s for c,s,_,_,_ in mrankC}
ordA=sorted(CANDIDATES,key=lambda c:-scA.get(c,0)); ordC=sorted(CANDIDATES,key=lambda c:-scC.get(c,0))
for br in band_recs:
    fq=[r["q"] for r in br["rings"] if not r["shared"]]
    if fq:
        ev=fds.score_phases(fq,fq,candidates=CANDIDATES,tol=FP_TOL,q_confirm_min=FP_QCONFIRM,rings=ERINGS)
        cf=[c for c in CANDIDATES if ev[c].verdict=="confirmed"]; br["ph"]=cf[0] if cf else None
print("=== 8b-3) Debye: 비정질(§4b, 느슨·저차) / 결정질(§4c, tight·전체) ===")
print(f"  전체 링 q={PKS or '없음'} | 결정구별용(nn외) q={PKC or '없음'}")
for c in ordC:
    vd=evi[c].verdict if evi else '-'
    print(f"  {c:7s} 비정질 match={scA.get(c,0):.3f} | 결정 match={scC.get(c,0):.3f} verdict={vd}")
_conf=[c for c in CANDIDATES if evi and evi[c].verdict=="confirmed"]
print(f"  → 결정(나노결정 포함) 확정: {_conf if _conf else '없음(nn외 고유링 부족)'} | 비정질만이면 nn~2A 공통이라 구별 약함")
fig,ax=plt.subplots(1,2,figsize=(14,4.4))
for p in PKS: ax[0].axvline(p,color=("0.6" if NN[0]<=p<=NN[1] else "tab:red"),lw=2.0,alpha=0.85)
ax[0].axvspan(*NN,color="0.9",alpha=0.6); ax[0].plot([],[],color="tab:red",lw=2,label="detected ring"); ax[0].plot([],[],color="0.6",lw=2,label="nn~2A (shared)")
_yo={c:0.9-0.16*i for i,c in enumerate(ordA)}
for c in ordA:
    for dd,w in _amrings[c]:
        qr=1/dd
        if 0.12<qr<1.1:
            on=any(abs(qr-p)<=AM_TOL for p in PKS); ax[0].plot([qr],[_yo[c]],marker="v",color=PHASE_COL[c],ms=6+6*w,alpha=0.95 if on else 0.5,mec="k" if on else "none")
    ax[0].plot([],[],marker="v",color=PHASE_COL[c],ls="",label=f"{c} ({scA.get(c,0):.2f})")
ax[0].set_ylim(0,1.05); ax[0].set_xlim(0.12,1.1); ax[0].set_yticks([]); ax[0].set_xlabel("q (1/A)")
ax[0].set_title(f"amorphous (§4b): peaks vs top-{AM_KEEP} electron-intensity rings"); ax[0].legend(fontsize=6.5,loc="upper right")
for p in PKS: ax[1].axvline(p,color=("0.6" if NN[0]<=p<=NN[1] else "tab:cyan"),lw=2.0,alpha=0.85)
ax[1].axvspan(*NN,color="0.9",alpha=0.6)
_yo2={c:0.9-0.16*i for i,c in enumerate(ordC)}
for c in ordC:
    for dd,w in ERINGS[c]:
        qr=1/dd
        if 0.15<qr<1.15:
            on=any(abs(qr-p)<=FP_TOL for p in PKS); ax[1].plot([qr],[_yo2[c]],marker="v",color=PHASE_COL[c],ms=6+5*w,alpha=1.0 if on else 0.3,mec="k" if on else "none")
    lbl=f"{c} [{evi[c].verdict}]" if (evi and evi[c].verdict!='weak/absent') else f"{c} ({scC.get(c,0):.2f})"
    ax[1].plot([],[],marker="v",color=PHASE_COL[c],ls="",label=lbl)
ax[1].set_ylim(0,1.05); ax[1].set_xlim(0.15,1.15); ax[1].set_yticks([]); ax[1].set_xlabel("q (1/A)")
ax[1].set_title("crystalline (§4c): peaks vs candidate crystal rings"); ax[1].legend(fontsize=6.5,loc="upper right")
fig.suptitle("8b-3) Debye match on FC-STEM rings -- amorphous(left)/crystalline(right); grey band = nn~2A shared by all",fontsize=9)
plt.tight_layout(); save(fig,"08b_fcstem_debye"); plt.show()

In [ ]:
# === 8b-4) STEP4b: 상 맵(결정확정 색칠) + §7 grain 합성 + CSV ===
import matplotlib.patches as mp
from matplotlib.colors import ListedColormap,BoundaryNorm
_fstk=np.stack([np.asarray(cm,float) for cm in cep_maps]); _dom=np.argmax(np.where(material[None,:,:],_fstk,-1e18),axis=0)
fcphase=np.full(scan,-1,int)
for i,br in enumerate(band_recs):
    if br["ph"] in CANDIDATES: fcphase[material&(tier<=1)&(_dom==i)]=CANDIDATES.index(br["ph"])
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
fig,ax=plt.subplots(1,2,figsize=(12,4.6))
ax[0].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
if (fcphase>=0).any(): ax[0].imshow(np.where(fcphase>=0,fcphase,np.nan),cmap=cmP,norm=nmP)
ax[0].set_title("STEP4 amorphous/nano phase (crystal-ring confirmed; grey=undetermined)"); ax[0].axis("off")
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
ax[1].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
if (fcphase>=0).any(): ax[1].imshow(np.where(fcphase>=0,fcphase,np.nan),cmap=cmP,norm=nmP,alpha=0.4)
ax[1].imshow(np.where(spot_tier>=2,spot_phase,np.nan),cmap=cmP,norm=nmP)
ax[1].set_title("combined: crystal grains (S7) + amorphous/nano (confirmed, faded)"); ax[1].axis("off")
fig.suptitle("8b-4) amorphous/nano phase + crystalline grains")
plt.tight_layout(); save(fig,"08b_fcstem_map"); plt.show()
_cnt={c:int((fcphase==k).sum()) for k,c in enumerate(CANDIDATES)}; _conf2={c:n for c,n in _cnt.items() if n}
print("  => 확정 비정질/나노 상 픽셀:",(_conf2 if _conf2 else "없음(nn외 고유링 부족)")," | 결정 grain(§7):",{CANDIDATES[k]:int(((spot_phase==k)&(spot_tier>=2)).sum()) for k in range(len(CANDIDATES)) if ((spot_phase==k)&(spot_tier>=2)).any()})
save_csv("08b_fcstem_bands",["r_A","strong_px","rings_q","d_A","radial_FWHM","azim_V","char","shared_nn","band_confirmed"],[[f"{br['d']:.2f}",int(br["strong"].sum()),";".join(f"{r['q']:.3f}" for r in br["rings"]),";".join(f"{r['d']:.2f}" for r in br["rings"]),";".join(f"{r.get('fwhm',float('nan')):.3f}" for r in br["rings"]),";".join(f"{r.get('V',0):.2f}" for r in br["rings"]),";".join(r.get("char","") for r in br["rings"]),";".join("nn" if r["shared"] else "-" for r in br["rings"]),br["ph"] or ""] for br in band_recs])

## 9) 두께 계산 — t/lambda = ln(I_total / I_beam)

앞에서 **물질/진공을 이미 판정**했으니 이를 활용한다. 진공(=물질 없는 곳)을 기준으로 dark를 자기교정하고 영점을 맞춘 **상대 두께**.
낮음=얇음(표면/껍데기), 높음=두꺼움. (수렴빔·각도분리라 절대 nm은 λ 필요 → 상대값만 신뢰.)

In [ ]:
# --- 이 셀 파라미터 ---
BEAM_RADIUS_PX=None    # 직접빔 디스크 반경(px). None=자동(~det/20)
VAC_PCTL=15            # 총세기 하위 X% = 엄격 기준진공(dark/영점용, 확실히 빈 곳). 물질이 많으면 낮추기
_flat=cube._flat_patterns(); _tot=np.asarray(_flat,float).reshape(_flat.shape[0],-1).sum(1)
vac_ref=np.asarray(_tot<=np.percentile(_tot,VAC_PCTL),bool).reshape(scan)   # 엄격 기준진공(오염 방지)
tmap,tex=fds.thickness_map(cube,center=center,beam_radius=BEAM_RADIUS_PX,vacuum_mask=vac_ref,return_extras=True)
tin=tmap[material]; tin=tin[np.isfinite(tin)]; tvac=tmap[vac_ref]; tvac=tvac[np.isfinite(tvac)]
print(f"dark 자기교정 D={tex['dark']:.3g} | 영점 offset={tex['offset']:.3g} | beam_r={tex['beam_radius']:.1f}px")
print(f"진공 t/lambda mean {tvac.mean():+.3f} (≈0 정상) | 물질 t/lambda mean {np.mean(tin):.2f} (5~95%: {np.percentile(tin,5):.2f}~{np.percentile(tin,95):.2f})")
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
im=ax[0].imshow(np.where(material,tmap,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[0].set_title("relative thickness t/lambda (vacuum-referenced)"); ax[0].axis("off")
ax[1].hist(tin,bins=50,color="0.4",label="material"); ax[1].axvline(0,color="r",ls="--",lw=1,label="vacuum (0)")
ax[1].set_xlabel("t/lambda"); ax[1].set_ylabel("# positions"); ax[1].set_title("thickness distribution (thin <-> thick)"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"09_thickness"); plt.show()
save_csv("09_thickness_stats",["metric","value"],
         [["dark",f"{tex['dark']:.4g}"],["vac_mean",f"{tvac.mean():.4f}"],["mat_mean",f"{np.mean(tin):.4f}"],
          ["mat_p05",f"{np.percentile(tin,5):.4f}"],["mat_p95",f"{np.percentile(tin,95):.4f}"],["material_frac",f"{material.mean():.4f}"]])
print("[정리] 확정/예상=스팟 인덱싱된 상 / 결정질(상불명)=링 신호는 진짜지만 상 미확정 / 비정질=halo만. 두께는 진공 대비 상대값.")